## NYC Property Sales Analysis — Starter
#### CIS 9655 Final Project | Group 19： Xixi Lin, Victor Louie, Jiahuan Wu, Chhin Lama


---

### Guiding Question

> Between 2000 and 2026, how do neighborhood-level factors such as
> crime rates, public school quality, and transportation accessibility
> influence residential property sale prices across New York City?


In [ ]:
####__________ dont need to run again if already install__________

# %pip install pandas
# !pip install pandas openpyxl xlrd
# %pip install plotly
# %pip install statsmodels
# %pip install dash
# %pip install dash plotly statsmodels
# %pip install pandas-ta
# %pip install openpyxl

# !{sys.executable} -m pip install openpyxl


In [2]:
"""
NYC Property Sales Analysis — Starter
CIS 9655 Final Project | Group 19
Data: property_data/ (borough .xls files, 2013–2025)
"""
import requests
import sys
import os
import glob
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, State, callback_context

# run import ....

### Step 1: Loading Data

In [3]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================

# ============================================================
# 1. PROPERTY_DATA - Combine all xls/xlsx files in property_data/
# ============================================================

property_folder = "Datasets/property_data"

all_property_files = glob.glob(os.path.join(property_folder, "*.xls")) + \
                     glob.glob(os.path.join(property_folder, "*.xlsx"))

df_list = []

def find_header_row(file, engine):
    """Find the row where BOROUGH appears — that's the header"""
    df_raw = pd.read_excel(file, engine=engine, header=None, nrows=15)
    for i, row in df_raw.iterrows():
        if row.astype(str).str.upper().str.contains("BOROUGH").any():
            return i
    return 4

for file in sorted(all_property_files):
    if ".DS_Store" in file:
        continue
    try:
        engine     = "xlrd" if file.endswith(".xls") else "openpyxl"
        header_row = find_header_row(file, engine)

        # ✅ USE skiprows= NOT header=
        df_temp = pd.read_excel(file, engine=engine, skiprows=header_row)

        # colums with _
        df_temp.columns = (df_temp.columns
                           .astype(str)
                           .str.strip()
                           .str.lower()
                           .str.replace(r"\s+", "_", regex=True)
                           .str.replace("\n", "_"))

        # Remove duplicate columns
        df_temp = df_temp.loc[:, ~df_temp.columns.duplicated()]

        basename = os.path.basename(file)
        name     = basename.replace(".xls", "").replace(".xlsx", "")
        parts    = name.split("_", 1)
        year     = parts[0]
        borough  = parts[1].replace("_", "")

        df_temp["year"]    = int(year)
        df_temp["borough"] = borough

        df_list.append(df_temp)
        print(f"✅ Loaded: {basename} (skiprows={header_row}) — {len(df_temp)} rows")

    except Exception as e:
        print(f"❌ Failed: {file} — {e}")

df_property = pd.concat(df_list, ignore_index=True)
print(f"\n📦 Total property rows: {len(df_property)}")
print(f"📅 Years: {sorted(df_property['year'].unique())}")

✅ Loaded: 2013_bronx.xls (skiprows=4) — 5932 rows
✅ Loaded: 2013_brooklyn.xls (skiprows=4) — 24842 rows
✅ Loaded: 2013_manhattan.xls (skiprows=4) — 26715 rows
✅ Loaded: 2013_queens.xls (skiprows=4) — 25415 rows
✅ Loaded: 2013_statenisland.xls (skiprows=4) — 7070 rows
✅ Loaded: 2014_bronx.xls (skiprows=4) — 6208 rows
✅ Loaded: 2014_brooklyn.xls (skiprows=4) — 25361 rows
✅ Loaded: 2014_manhattan.xls (skiprows=4) — 24524 rows
✅ Loaded: 2014_queens.xls (skiprows=4) — 25157 rows
✅ Loaded: 2014_statenisland.xls (skiprows=4) — 7437 rows
✅ Loaded: 2015_bronx.xls (skiprows=4) — 6736 rows
✅ Loaded: 2015_brooklyn.xls (skiprows=4) — 25362 rows
✅ Loaded: 2015_manhattan.xls (skiprows=4) — 24989 rows
✅ Loaded: 2015_queens.xls (skiprows=4) — 26354 rows
✅ Loaded: 2015_statenisland.xls (skiprows=4) — 8005 rows
✅ Loaded: 2016_bronx.xls (skiprows=4) — 7203 rows
✅ Loaded: 2016_brooklyn.xls (skiprows=4) — 25519 rows
✅ Loaded: 2016_manhattan.xls (skiprows=4) — 21241 rows
✅ Loaded: 2016_queens.xls (skiprows=4

In [4]:
print(df_property.groupby("year").size().to_string())

year
2013    89974
2014    88687
2015    91446
2016    89806
2017    88660
2018    83782
2019    83920
2020    68689
2021    99095
2022    93432
2023    76416
2024    78248
2025    84936


In [5]:
# ============================================================
# 2. CRIME DATA
# ============================================================

df_arrest    = pd.read_csv("Datasets/NYPD_Arrest_Data_(Year_to_Date)_20260228.csv")
df_spotcrime = pd.read_csv("Datasets/spotcrime.crime.ny.csv")

print(f"✅ NYPD Arrest Data: {len(df_arrest)} rows")
print(f"✅ SpotCrime Data: {len(df_spotcrime)} rows")


✅ NYPD Arrest Data: 278953 rows
✅ SpotCrime Data: 61824 rows


In [6]:
# ============================================================
# 3. SCHOOL & INCOME DATA
# ============================================================
# give up the old school fils
# df_schools = pd.read_excel("Datasets/NYC Public School Rankings 2025.xlsx", engine="openpyxl")
df_income  = pd.read_excel("Datasets/NYC Median Household Income.xlsx",     engine="openpyxl")

#new added school fiels
df_elem   = pd.read_excel("Datasets/NYC_Top_100_Elementary_Schools.xlsx", engine="openpyxl")
df_high   = pd.read_excel("Datasets/NYC_Top_100_Public_High_Schools.xlsx", engine="openpyxl")
df_middle = pd.read_excel("Datasets/NYC_Top_Middle_Schools.xlsx", engine="openpyxl")

#print(f"✅ School Rankings: {len(df_schools)} rows")
print(f"✅ Median Income:   {len(df_income)} rows")
print(f"✅ Elementary      : {len(df_elem)} rows")
print(f"✅ High Schools    : {len(df_high)} rows")
print(f"✅ Middle Schools  : {len(df_middle)} rows")


✅ Median Income:   56 rows
✅ Elementary      : 40 rows
✅ High Schools    : 100 rows
✅ Middle Schools  : 40 rows


In [7]:
# ============================================================
# 4. TRANSPORTATION DATA
# ============================================================

df_subway = pd.read_csv("Datasets/MTA_Subway_Stations_20260228.csv")
print(f"✅ Subway Stations: {len(df_subway)} rows")


✅ Subway Stations: 496 rows


In [8]:
# ============================================================
# 5. NYC PROPERTY SALES (main CSV)
# ============================================================

df_sales = pd.read_csv("Datasets/NYC_Property_Sales.csv")
print(f"✅ NYC Property Sales CSV: {len(df_sales)} rows")


✅ NYC Property Sales CSV: 57239 rows


In [9]:
# ============================================================
# 6. CS REPORTS (quarterly)
# ============================================================

cs_files   = glob.glob("Datasets/cs-report-q*.xlsx")
df_cs_list = [pd.read_excel(f, engine="openpyxl") for f in sorted(cs_files)]
df_cs      = pd.concat(df_cs_list, ignore_index=True)
print(f"✅ CS Reports combined: {len(df_cs)} rows")

✅ CS Reports combined: 324 rows


## Step 2: Clean data

In [10]:
# ============================================================
# 2A. CLEAN df_property
# ============================================================

df_property = df_property.loc[:, ~df_property.columns.str.startswith("unnamed")]
df_property = df_property.loc[:, ~df_property.columns.duplicated()]
df_property = df_property.dropna(subset=["borough"])

# Fix borough names — including 2018+ files with extra 'x'
borough_fix_map = {
    "bronx"         : "bronx",
    "brooklyn"      : "brooklyn",
    "manhattan"     : "manhattan",
    "queens"        : "queens",
    "statenisland"  : "statenisland",
    "staten_island" : "statenisland",
    "bronxx"        : "bronx",
    "brooklynx"     : "brooklyn",
    "manhattanx"    : "manhattan",
    "queensx"       : "queens",
    "statenislandx" : "statenisland"
}
df_property["borough"] = (df_property["borough"]
                           .str.strip()
                           .str.lower()
                           .map(borough_fix_map))

# Fix zip code
df_property["zip_code"] = (df_property["zip_code"]
                            .astype(str)
                            .str.replace(".0", "", regex=False)
                            .str.strip()
                            .str.replace("nan", "00000")
                            .str.zfill(5))

# Convert sale_price
df_property["sale_price"] = pd.to_numeric(df_property["sale_price"], errors="coerce")
df_property = df_property[df_property["sale_price"] > 0]

# Convert sale_date
df_property["sale_date"] = pd.to_datetime(df_property["sale_date"], errors="coerce")

# Drop missing critical fields
df_property = df_property.dropna(subset=["sale_price", "sale_date", "borough"])
df_property = df_property.reset_index(drop=True)

print(f"✅ df_property cleaned: {df_property.shape}")
print(f"   Boroughs : {sorted(df_property['borough'].unique())}")
print(f"   Years    : {sorted(df_property['year'].unique())}")
print(f"   ZIP sample: {df_property['zip_code'].head(3).tolist()}")

✅ df_property cleaned: (772459, 26)
   Boroughs : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
   Years    : [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
   ZIP sample: ['10457', '10458', '10458']


In [11]:
# ============================================================
# 2B-1. CLEAN df_sales
# ============================================================

# Rename columns to lowercase
df_sales.columns = (df_sales.columns
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .str.replace(r"\s+", "_", regex=True))

# Map borough numbers to names
boro_num_map = {1: "manhattan", 2: "bronx", 3: "brooklyn",
                4: "queens",    5: "statenisland"}
df_sales["borough"] = pd.to_numeric(df_sales["borough"], errors="coerce").map(boro_num_map)

# Fix zip code
df_sales["zip_code"] = (df_sales["zip_code"]
                         .astype(str)
                         .str.replace(".0", "", regex=False)
                         .str.strip()
                         .str.replace("nan", "00000")
                         .str.zfill(5))

# Convert sale_price
df_sales["sale_price"] = pd.to_numeric(df_sales["sale_price"], errors="coerce")
df_sales = df_sales[df_sales["sale_price"] > 0]

# Convert sale_date
df_sales["sale_date"] = pd.to_datetime(df_sales["sale_date"], errors="coerce")

# Drop missing critical fields
df_sales = df_sales.dropna(subset=["sale_price", "sale_date", "borough"])
df_sales = df_sales.reset_index(drop=True)

print(f"✅ df_sales cleaned: {df_sales.shape}")
print(f"   Boroughs : {sorted(df_sales['borough'].unique())}")
print(f"   ZIP sample: {df_sales['zip_code'].head(3).tolist()}")

✅ df_sales cleaned: (57239, 19)
   Boroughs : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
   ZIP sample: ['10457', '10457', '10457']


In [12]:
# ============================================================
# 2B-2. CLEAN INVALID PRICES
# Only remove clearly invalid transactions
# Keep all real market prices including high-end Manhattan properties
# ============================================================

print("=== sale_price Statistics (BEFORE cleaning) ===")
print(df_property["sale_price"].describe())
print(f"\nMin   : ${df_property['sale_price'].min():,.0f}")
print(f"Max   : ${df_property['sale_price'].max():,.0f}")
print(f"Mean  : ${df_property['sale_price'].mean():,.0f}")
print(f"Median: ${df_property['sale_price'].median():,.0f}")

# Remove only clearly invalid transactions (not real market sales)
# $10,000 threshold covers: $0 transfers, $1 gifts, clerical errors
invalid_before = (df_property["sale_price"] < 10_000).sum()

df_property = df_property[df_property["sale_price"] >= 10_000].reset_index(drop=True)
df_sales    = df_sales[df_sales["sale_price"]    >= 10_000].reset_index(drop=True)

print(f"\n=== After Cleaning ===")
print(f"Removed {invalid_before:,} invalid transactions (price < $10,000)")
print(f"✅ df_property : {df_property.shape}")
print(f"✅ df_sales    : {df_sales.shape}")
print(f"\n=== sale_price Statistics (AFTER cleaning) ===")
print(df_property["sale_price"].describe())

=== sale_price Statistics (BEFORE cleaning) ===
count    7.724590e+05
mean     1.876735e+06
std      1.514884e+07
min      1.000000e+00
25%      3.900000e+05
50%      6.773780e+05
75%      1.190000e+06
max      4.111112e+09
Name: sale_price, dtype: float64

Min   : $1
Max   : $4,111,111,766
Mean  : $1,876,735
Median: $677,378

=== After Cleaning ===
Removed 20,134 invalid transactions (price < $10,000)
✅ df_property : (752325, 26)
✅ df_sales    : (55948, 19)

=== sale_price Statistics (AFTER cleaning) ===
count    7.523250e+05
mean     1.926945e+06
std      1.534706e+07
min      1.000000e+04
25%      4.100000e+05
50%      6.950000e+05
75%      1.200000e+06
max      4.111112e+09
Name: sale_price, dtype: float64


In [13]:
# ============================================================
# 2C. CLEAN df_arrest (NYPD)
# Columns: ARREST_KEY, ARREST_DATE, OFNS_DESC, LAW_CAT_CD,
#          ARREST_BORO, Latitude, Longitude
# ============================================================

# Keep only relevant columns
df_arrest = df_arrest[["ARREST_DATE", "OFNS_DESC", "LAW_CAT_CD",
                        "ARREST_BORO", "Latitude", "Longitude"]].copy()

# Convert date
df_arrest["ARREST_DATE"] = pd.to_datetime(df_arrest["ARREST_DATE"], errors="coerce")

# Map borough codes to names
boro_code_map = {"B": "bronx", "K": "brooklyn", "M": "manhattan",
                 "Q": "queens",  "S": "statenisland"}
df_arrest["borough"] = df_arrest["ARREST_BORO"].map(boro_code_map)

df_arrest = df_arrest.dropna(subset=["Latitude", "Longitude", "borough"])
df_arrest = df_arrest.reset_index(drop=True)

print(f"✅ df_arrest cleaned: {df_arrest.shape}")
print(f"   Boroughs: {sorted(df_arrest['borough'].unique())}")

✅ df_arrest cleaned: (278953, 7)
   Boroughs: ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']


In [14]:
# ============================================================
# 2D. CLEAN df_spotcrime
# Columns: CrimeType, CrimeAddress, CrimeTime, Latitude, Longitude, City
# ============================================================

df_spotcrime = df_spotcrime[["CrimeType", "CrimeAddress", "CrimeTime",
                              "Latitude", "Longitude", "City"]].copy()

df_spotcrime["CrimeTime"] = pd.to_datetime(df_spotcrime["CrimeTime"], errors="coerce")
df_spotcrime = df_spotcrime.dropna(subset=["Latitude", "Longitude"])
df_spotcrime = df_spotcrime.reset_index(drop=True)

print(f"✅ df_spotcrime cleaned: {df_spotcrime.shape}")


✅ df_spotcrime cleaned: (61824, 6)


In [15]:
# ============================================================
# 2E. COMBINE & CLEAN SCHOOL DATA (Elementary + High + Middle)
# ============================================================

def clean_school_file(df, level):
    df = df.copy()
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    # Rename columns to be consistent
    df = df.rename(columns={
        "school"    : "school_name",
        "school_name": "school_name",
        "area_(borough)": "borough",
        "borough"   : "borough",
        "zip_code"  : "zip_code"
    })
    # Clean zip_code
    df["zip_code"] = df["zip_code"].astype(str).str.strip().str.zfill(5)
    # Clean borough
    df["borough"] = (df["borough"]
                     .str.strip().str.lower()
                     .str.replace(" ", "")
                     .str.replace("statenisland", "statenisland"))
    # Add school level label
    df["school_level"] = level
    # Keep only needed columns
    return df[["rank", "school_name", "borough", "zip_code", "type", "school_level"]]

df_elem   = clean_school_file(df_elem,   "elementary")
df_high   = clean_school_file(df_high,   "high")
df_middle = clean_school_file(df_middle, "middle")

# Combine all 3 into one table
df_schools_all = pd.concat([df_elem, df_high, df_middle], ignore_index=True)
df_schools_all = df_schools_all.drop_duplicates(subset=["school_name"]).reset_index(drop=True)

print(f"✅ df_schools_all combined: {df_schools_all.shape}")
print(f"   Levels   : {sorted(df_schools_all['school_level'].unique())}")
print(f"   Boroughs : {sorted(df_schools_all['borough'].unique())}")
print(f"\nSample:")
print(df_schools_all.head(6))

✅ df_schools_all combined: (178, 6)
   Levels   : ['elementary', 'high', 'middle']
   Boroughs : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']

Sample:
   rank                     school_name    borough zip_code     type  \
0     1  Success Academy Hell’s Kitchen  manhattan    10019  Charter
1     2                          NEST+m  manhattan    10002   Public
2     3     Success Academy Bensonhurst   brooklyn    11214  Charter
3     4          Icahn Charter School 6      bronx    10457  Charter
4     5        Success Academy Bushwick   brooklyn    11221  Charter
5     6    Success Academy Hudson Yards  manhattan    10001  Charter

  school_level
0   elementary
1   elementary
2   elementary
3   elementary
4   elementary
5   elementary


In [16]:
# ============================================================
# 2F. CLEAN df_income
# Columns: Rank, ZIP Code, USD
# ============================================================

df_income.columns = ["rank", "zip_code", "median_income_usd"]

# Clean zip code — remove ', NY'
df_income["zip_code"] = (df_income["zip_code"]
                          .str.replace(", NY", "")
                          .str.strip()
                          .str.zfill(5))
df_income["median_income_usd"] = pd.to_numeric(df_income["median_income_usd"], errors="coerce")

df_income = df_income.dropna()
df_income = df_income.reset_index(drop=True)

print(f"✅ df_income cleaned: {df_income.shape}")
print(f"   ZIP sample: {df_income['zip_code'].head(3).tolist()}")


✅ df_income cleaned: (56, 3)
   ZIP sample: ['10282', '10007', '10069']


In [17]:
# ============================================================
# 2G. RELOAD AND CLEAN df_subway from scratch
# ============================================================

# Reload the original file
df_subway = pd.read_csv("Datasets/MTA_Subway_Stations_20260228.csv")

# Keep only relevant columns
df_subway = df_subway[["Station ID", "Stop Name", "Borough",
                        "GTFS Latitude", "GTFS Longitude"]].copy()

# Rename columns
df_subway.columns = ["station_id", "stop_name", "borough",
                     "latitude", "longitude"]

# Standardize borough codes to full names
df_subway["borough"] = df_subway["borough"].str.strip().str.lower()

subway_boro_map = {
    "bk": "brooklyn",
    "bx": "bronx",
    "m" : "manhattan",
    "q" : "queens",
    "si": "statenisland"
}
df_subway["borough"] = df_subway["borough"].map(subway_boro_map)

df_subway = df_subway.dropna(subset=["latitude", "longitude", "borough"])
df_subway = df_subway.reset_index(drop=True)

print(f"✅ df_subway cleaned: {df_subway.shape}")
print(f"   Boroughs: {sorted(df_subway['borough'].unique())}")

✅ df_subway cleaned: (496, 5)
   Boroughs: ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']


In [18]:
# Check all prices above $100M in original data
#！！！！！！！！！！！！！！！！！！！！
print("=== Prices above $100,000,000 in df_property ===")
high_price = df_property[df_property["sale_price"] > 100_000_000]
print(f"Total rows: {len(high_price)}")
print(high_price[["borough", "zip_code", "sale_price", "sale_date", "address"]].sort_values("sale_price", ascending=False).to_string())

print("\n=== Prices above $100,000,000 in df_sales ===")
high_price_sales = df_sales[df_sales["sale_price"] > 100_000_000]
print(f"Total rows: {len(high_price_sales)}")
print(high_price_sales[["borough", "zip_code", "sale_price", "sale_date"]].sort_values("sale_price", ascending=False).to_string())

=== Prices above $100,000,000 in df_property ===
Total rows: 1317
             borough zip_code    sale_price  sale_date                                    address
145900     manhattan    10009  4.111112e+09 2015-12-18  240 1 AVENUE
85767      manhattan    10009  3.330133e+09 2014-06-03  240 1 AVENUE
325646     manhattan    10011  2.397502e+09 2018-03-20                                75 9 AVENUE
270831     manhattan    10167  2.210000e+09 2017-05-05                            245 PARK AVENUE
149431     manhattan    10036  2.200000e+09 2015-01-15  1095 AVENUE OF THE AMER
386345     manhattan    10001  2.155000e+09 2019-06-14                        30 HUDSON YARDS, TW
86703      manhattan    10022  2.100000e+09 2014-12-03  611 5 AVENUE
211414     manhattan    10019  1.932900e+09 2016-01-27  787 7 AVENUE
146854     manhattan    10022  1.791829e+09 2015-02-10  301 PARK AVENUE
148839     manhattan    10019  1.775000e+09 2015-04-17  730 5 AVENUE
214506     manhattan    10013  1.767913e+09 2

In [19]:
# Check what building classes exist
print(df_property["building_class_category"].value_counts().head(20))

building_class_category
01 ONE FAMILY DWELLINGS                         92513
10 COOPS - ELEVATOR APARTMENTS                  89334
13 CONDOS - ELEVATOR APARTMENTS                 78881
02 TWO FAMILY DWELLINGS                         66929
10  COOPS - ELEVATOR APARTMENTS                 50873
01  ONE FAMILY DWELLINGS                        45967
13  CONDOS - ELEVATOR APARTMENTS                38053
02  TWO FAMILY DWELLINGS                        37940
09 COOPS - WALKUP APARTMENTS                    18511
03 THREE FAMILY DWELLINGS                       16146
01 ONE FAMILY DWELLINGS                         13642
10 COOPS - ELEVATOR APARTMENTS                  11883
07 RENTALS - WALKUP APARTMENTS                  11751
13 CONDOS - ELEVATOR APARTMENTS                 10769
09  COOPS - WALKUP APARTMENTS                   10480
02 TWO FAMILY DWELLINGS                         10256
03  THREE FAMILY DWELLINGS                       9922
15 CONDOS - 2-10 UNIT RESIDENTIAL                9705
07  

In [20]:
# ============================================================
# 2H. PRICE VALIDATION CHECK
# Remove clearly invalid prices on both ends
# ============================================================

PRICE_MIN = 10_000        # below this = not a real market transaction
PRICE_MAX = 500_000_000   # above this = data entry error

invalid_low_prop  = (df_property["sale_price"] <  PRICE_MIN).sum()
invalid_high_prop = (df_property["sale_price"] >  PRICE_MAX).sum()
invalid_low_sale  = (df_sales["sale_price"]    <  PRICE_MIN).sum()
invalid_high_sale = (df_sales["sale_price"]    >  PRICE_MAX).sum()

# Apply to df_property
df_property = df_property[
    (df_property["sale_price"] >= PRICE_MIN) &
    (df_property["sale_price"] <= PRICE_MAX)
].reset_index(drop=True)

# Apply to df_sales
df_sales = df_sales[
    (df_sales["sale_price"] >= PRICE_MIN) &
    (df_sales["sale_price"] <= PRICE_MAX)
].reset_index(drop=True)

print(f"df_property — removed {invalid_low_prop:,} below ${PRICE_MIN:,} and {invalid_high_prop:,} above ${PRICE_MAX:,}")
print(f"df_sales    — removed {invalid_low_sale:,} below ${PRICE_MIN:,} and {invalid_high_sale:,} above ${PRICE_MAX:,}")
print(f"\n✅ df_property : {df_property.shape}")
print(f"✅ df_sales    : {df_sales.shape}")

print(f"\n=== Price range by borough ===")
for boro in sorted(df_property["borough"].dropna().unique()):
    df_b = df_property[df_property["borough"] == boro]
    print(f"{boro:15} | median: ${df_b['sale_price'].median():>12,.0f} | max: ${df_b['sale_price'].max():>12,.0f}")

df_property — removed 0 below $10,000 and 79 above $500,000,000
df_sales    — removed 0 below $10,000 and 2 above $500,000,000

✅ df_property : (752246, 26)
✅ df_sales    : (55946, 19)

=== Price range by borough ===
bronx           | median: $     476,250 | max: $ 133,400,000
brooklyn        | median: $     818,300 | max: $ 345,000,000
manhattan       | median: $   1,095,000 | max: $ 498,450,000
queens          | median: $     570,000 | max: $ 475,000,000
statenisland    | median: $     525,000 | max: $ 135,259,627


In [21]:
# ============================================================
# 2H-2. FILTER RESIDENTIAL PROPERTIES ONLY
# Remove commercial, industrial, and large building transactions
# Keep only real residential property sales
# ============================================================

# Define residential building class categories
residential_keywords = [
    "ONE FAMILY",
    "TWO FAMILY",
    "THREE FAMILY",
    "CONDO",
    "COOP",
    "RENTAL",
    "RENTALS",
    "RESIDENTI"
]

# Clean building_class_category column first
df_property["building_class_category"] = (
    df_property["building_class_category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_sales["building_class_category"] = (
    df_sales["building_class_category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Build filter mask
def is_residential(cat):
    return any(kw in str(cat).upper() for kw in residential_keywords)

mask_property = df_property["building_class_category"].apply(is_residential)
mask_sales    = df_sales["building_class_category"].apply(is_residential)

before_prop  = len(df_property)
before_sales = len(df_sales)

df_property = df_property[mask_property].reset_index(drop=True)
df_sales    = df_sales[mask_sales].reset_index(drop=True)

print(f"df_property — removed {before_prop - len(df_property):,} non-residential rows")
print(f"df_sales    — removed {before_sales - len(df_sales):,} non-residential rows")
print(f"\n✅ df_property : {df_property.shape}")
print(f"✅ df_sales    : {df_sales.shape}")

print(f"\n=== Price range by borough AFTER residential filter ===")
for boro in sorted(df_property["borough"].dropna().unique()):
    df_b = df_property[df_property["borough"] == boro]
    print(f"{boro:15} | median: ${df_b['sale_price'].median():>12,.0f} | max: ${df_b['sale_price'].max():>12,.0f}")

df_property — removed 35,874 non-residential rows
df_sales    — removed 2,353 non-residential rows

✅ df_property : (716372, 26)
✅ df_sales    : (53593, 19)

=== Price range by borough AFTER residential filter ===
bronx           | median: $     460,000 | max: $ 133,400,000
brooklyn        | median: $     800,000 | max: $ 317,792,634
manhattan       | median: $   1,096,000 | max: $ 487,500,000
queens          | median: $     557,500 | max: $ 475,000,000
statenisland    | median: $     525,000 | max: $ 135,259,627


In [22]:
# Check what these super high price residential properties are
print("=== Top 20 highest price properties after residential filter ===")
print(df_property.nlargest(20, "sale_price")[
    ["borough", "zip_code", "sale_price", "sale_date",
     "address", "building_class_category", "total_units"]
].to_string())

=== Top 20 highest price properties after residential filter ===
          borough zip_code   sale_price  sale_date                                    address                    building_class_category  total_units
530157  manhattan    10038  487500000.0 2022-09-23                            19 DUTCH STREET           08 RENTALS - ELEVATOR APARTMENTS        485.0
332614     queens    11375  475000000.0 2018-11-08                    104-20 QUEENS BOULEVARD           08 RENTALS - ELEVATOR APARTMENTS       1348.0
134073  manhattan    10010  471753283.0 2015-08-18  11 MADISON AVENUE                                         43  CONDO OFFICE BUILDINGS          1.0
313862  manhattan    10036  442000000.0 2018-09-21                              1535 BROADWAY  42 CONDO CULTURAL/MEDICAL/EDUCATIONAL/ETC          0.0
591739  manhattan    10019  425000000.0 2023-12-19                            724 5 AVENUE, B                  43 CONDO OFFICE BUILDINGS          1.0
591740  manhattan    10019  4250000

In [23]:
# ============================================================
# 2H-3. FILTER SINGLE RESIDENTIAL UNITS
# Remove entire apartment buildings and commercial condos
# Keep only individual residential unit transactions
# ============================================================

# Remove office/commercial condo categories
commercial_keywords = [
    "OFFICE",
    "STORE",
    "CULTURAL",
    "MEDICAL",
    "EDUCATIONAL",
    "PARKING",
    "SPECIAL CONDO BILLING"
]

def is_not_commercial(cat):
    return not any(kw in str(cat).upper() for kw in commercial_keywords)

# Also filter out entire buildings (total_units > 10)
mask_prop = (
    df_property["building_class_category"].apply(is_not_commercial) &
    (pd.to_numeric(df_property["total_units"], errors="coerce") <= 10)
)

mask_sales = (
    df_sales["building_class_category"].apply(is_not_commercial) &
    (pd.to_numeric(df_sales["total_units"], errors="coerce") <= 10)
)

before_prop  = len(df_property)
before_sales = len(df_sales)

df_property = df_property[mask_prop].reset_index(drop=True)
df_sales    = df_sales[mask_sales].reset_index(drop=True)

print(f"df_property — removed {before_prop - len(df_property):,} rows")
print(f"df_sales    — removed {before_sales - len(df_sales):,} rows")
print(f"\n✅ df_property : {df_property.shape}")
print(f"✅ df_sales    : {df_sales.shape}")

print(f"\n=== Price range by borough AFTER unit filter ===")
for boro in sorted(df_property["borough"].dropna().unique()):
    df_b = df_property[df_property["borough"] == boro]
    print(f"{boro:15} | median: ${df_b['sale_price'].median():>12,.0f} | max: ${df_b['sale_price'].max():>12,.0f}")

df_property — removed 135,099 rows
df_sales    — removed 11,591 rows

✅ df_property : (581273, 26)
✅ df_sales    : (42002, 19)

=== Price range by borough AFTER unit filter ===
bronx           | median: $     470,000 | max: $ 107,357,142
brooklyn        | median: $     827,500 | max: $ 317,792,634
manhattan       | median: $   1,195,425 | max: $ 368,571,302
queens          | median: $     605,000 | max: $ 161,342,986
statenisland    | median: $     525,000 | max: $  34,714,978


In [24]:
# Check top 10 highest price after all filters
print("=== Top 10 highest price after all filters ===")
print(df_property.nlargest(10, "sale_price")[
    ["borough", "zip_code", "sale_price", "sale_date",
     "address", "building_class_category", "total_units",
     "residential_units"]
].to_string())

=== Top 10 highest price after all filters ===
          borough zip_code   sale_price  sale_date                                    address building_class_category  total_units  residential_units
81075   manhattan    10019  368571302.0 2014-08-05  157 WEST 57TH   STREET                           45  CONDO HOTELS          1.0                0.0
305640  manhattan    10019  366170000.0 2018-11-30                               870 7 AVENUE         45 CONDO HOTELS          1.0                0.0
469292  manhattan    10014  340000000.0 2022-07-08               532 WASHINGTON STREET, UNIT-    28 COMMERCIAL CONDOS          1.0                NaN
25998   manhattan    10036  325000000.0 2013-03-08  700 8 AVENUE                                     45  CONDO HOTELS          1.0                0.0
494435   brooklyn    11201  317792634.0 2023-11-28                     85 SMITH STREET, HOTEL         45 CONDO HOTELS          1.0                NaN
351993  manhattan    10022  298000000.0 2019-10-31   

In [25]:
# ============================================================
# 2H-4. REMOVE HOTELS AND COMMERCIAL CONDOS
# Keep only true residential properties
# ============================================================

hotel_commercial_keywords = [
    "HOTEL",
    "COMMERCIAL CONDO"
]

def is_not_hotel(cat):
    return not any(kw in str(cat).upper() for kw in hotel_commercial_keywords)

before_prop  = len(df_property)
before_sales = len(df_sales)

# Remove hotels and commercial condos
# Also require at least 1 residential unit
df_property = df_property[
    df_property["building_class_category"].apply(is_not_hotel) &
    (pd.to_numeric(df_property["residential_units"], errors="coerce") >= 1)
].reset_index(drop=True)

df_sales = df_sales[
    df_sales["building_class_category"].apply(is_not_hotel) &
    (pd.to_numeric(df_sales["residential_units"], errors="coerce") >= 1)
].reset_index(drop=True)

print(f"df_property — removed {before_prop - len(df_property):,} rows")
print(f"df_sales    — removed {before_sales - len(df_sales):,} rows")
print(f"\n✅ df_property : {df_property.shape}")
print(f"✅ df_sales    : {df_sales.shape}")

print(f"\n=== Price range by borough AFTER hotel filter ===")
for boro in sorted(df_property["borough"].dropna().unique()):
    df_b = df_property[df_property["borough"] == boro]
    print(f"{boro:15} | median: ${df_b['sale_price'].median():>12,.0f} | max: ${df_b['sale_price'].max():>12,.0f}")

print(f"\n=== Top 5 highest price after all filters ===")
print(df_property.nlargest(5, "sale_price")[
    ["borough", "zip_code", "sale_price", "address",
     "building_class_category", "residential_units"]
].to_string())

df_property — removed 109,842 rows
df_sales    — removed 1,042 rows

✅ df_property : (471431, 26)
✅ df_sales    : (40960, 19)

=== Price range by borough AFTER hotel filter ===
bronx           | median: $     510,000 | max: $  87,400,000
brooklyn        | median: $     880,000 | max: $  68,000,000
manhattan       | median: $   1,665,000 | max: $ 239,958,219
queens          | median: $     675,000 | max: $ 161,342,986
statenisland    | median: $     527,000 | max: $  34,714,978

=== Top 5 highest price after all filters ===
          borough zip_code   sale_price                                    address          building_class_category  residential_units
246276  manhattan    10019  239958219.0                 220 CENTRAL PARK SOUTH, 50  13 CONDOS - ELEVATOR APARTMENTS                1.0
247346  manhattan    10065  200000000.0                     165 EAST 66TH ST, RESI  13 CONDOS - ELEVATOR APARTMENTS                1.0
316237  manhattan    10019  188000000.0                220 CENTRAL

In [26]:
print(df_property["borough"].value_counts())

borough
queens          147046
brooklyn        139389
manhattan        83015
statenisland     63135
bronx            38846
Name: count, dtype: int64


In [27]:
# ============================================================
# 2I. CHECK DUPLICATES
# ============================================================

# df_property
print(f"=== df_property ===")
print(f"Total rows      : {len(df_property):,}")
print(f"Duplicates      : {df_property.duplicated().sum():,}")
print(f"Unique rows     : {len(df_property) - df_property.duplicated().sum():,}")

print(f"\nSample duplicate rows:")
print(df_property[df_property.duplicated(keep=False)]
      [["borough", "zip_code", "sale_price", "sale_date", "year"]]
      .sort_values("sale_price")
      .head(10))

# df_sales
print(f"\n=== df_sales ===")
print(f"Total rows      : {len(df_sales):,}")
print(f"Duplicates      : {df_sales.duplicated().sum():,}")
print(f"Unique rows     : {len(df_sales) - df_sales.duplicated().sum():,}")

=== df_property ===
Total rows      : 471,431
Duplicates      : 431
Unique rows     : 471,000

Sample duplicate rows:
         borough zip_code  sale_price  sale_date  year
2744    brooklyn    11209     10000.0 2013-11-15  2013
65112     queens    11417     10000.0 2014-03-17  2014
2743    brooklyn    11209     10000.0 2013-11-15  2013
65113     queens    11417     10000.0 2014-03-17  2014
140600    queens    11429     10000.0 2016-11-28  2016
140601    queens    11429     10000.0 2016-11-28  2016
44067   brooklyn    11208     11350.0 2014-05-08  2014
44066   brooklyn    11208     11350.0 2014-05-08  2014
119189  brooklyn    11213     11750.0 2016-09-07  2016
119190  brooklyn    11213     11750.0 2016-09-07  2016

=== df_sales ===
Total rows      : 40,960
Duplicates      : 0
Unique rows     : 40,960


In [28]:
# ============================================================
# 2I. REMOVE DUPLICATES
# ============================================================

before = len(df_property)
df_property = df_property.drop_duplicates()
df_property = df_property.reset_index(drop=True)
print(f"df_property — Before: {before:,} → After: {len(df_property):,} (Removed: {before - len(df_property):,})")

before = len(df_sales)
df_sales = df_sales.drop_duplicates()
df_sales = df_sales.reset_index(drop=True)
print(f"df_sales    — Before: {before:,} → After: {len(df_sales):,} (Removed: {before - len(df_sales):,})")

df_property — Before: 471,431 → After: 471,000 (Removed: 431)
df_sales    — Before: 40,960 → After: 40,960 (Removed: 0)


In [29]:
# ============================================================
# 2J. CHECK & CLEAN YEAR BUILT
# ============================================================

print("=== year_built Statistics (BEFORE) ===")
print(df_property["year_built"].describe())
print(f"\nUnrealistic year_built = 0   : {(df_property['year_built'] == 0).sum():,}")
print(f"Unrealistic year_built > 2025: {(df_property['year_built'] > 2025).sum():,}")
print(f"Unrealistic year_built < 1800: {(df_property['year_built'] < 1800).sum():,}")
print(f"Missing year_built           : {df_property['year_built'].isna().sum():,}")
print(f"\nYear built distribution (top 10):")
print(df_property["year_built"].value_counts().head(10))

# --- Clean ---
before_zeros   = (df_property["year_built"] == 0).sum()
before_future  = (df_property["year_built"] > 2025).sum()
before_ancient = (df_property["year_built"] < 1800).sum()

df_property["year_built"] = pd.to_numeric(df_property["year_built"], errors="coerce")
df_property.loc[df_property["year_built"] == 0,   "year_built"] = np.nan
df_property.loc[df_property["year_built"] > 2025, "year_built"] = np.nan
df_property.loc[df_property["year_built"] < 1800, "year_built"] = np.nan

print(f"\n=== year_built Statistics (AFTER) ===")
print(f"Replaced {before_zeros:,} zeros with NaN")
print(f"Replaced {before_future:,} future years with NaN")
print(f"Replaced {before_ancient:,} ancient years (<1800) with NaN")
print(f"Total missing year_built: {df_property['year_built'].isna().sum():,}")
print(f"Min: {df_property['year_built'].min()}")
print(f"Max: {df_property['year_built'].max()}")

=== year_built Statistics (BEFORE) ===
count    455020.000000
mean       1884.101312
std         365.663510
min           0.000000
25%        1920.000000
50%        1940.000000
75%        1989.000000
max        2025.000000
Name: year_built, dtype: float64

Unrealistic year_built = 0   : 16,342
Unrealistic year_built > 2025: 0
Unrealistic year_built < 1800: 16,348
Missing year_built           : 15,980

Year built distribution (top 10):
year_built
1920.0    40016
1930.0    31618
1925.0    29519
1910.0    20444
1950.0    18601
0.0       16342
1940.0    15431
1960.0    14678
1899.0    13189
1901.0    12721
Name: count, dtype: int64

=== year_built Statistics (AFTER) ===
Replaced 16,342 zeros with NaN
Replaced 0 future years with NaN
Replaced 16,348 ancient years (<1800) with NaN
Total missing year_built: 32,328
Min: 1800.0
Max: 2025.0


In [30]:
# ============================================================
# 2K. CHECK & CLEAN GROSS SQUARE FEET
# ============================================================

print("=== gross_square_feet Statistics (BEFORE) ===")
print(df_property["gross_square_feet"].describe())

# Count zeros
before_zeros = (df_property["gross_square_feet"] == 0).sum()
print(f"\ngross_square_feet = 0: {before_zeros:,} rows")
print(f"Missing              : {df_property['gross_square_feet'].isna().sum():,}")

# Replace 0 with NaN
df_property["gross_square_feet"] = pd.to_numeric(df_property["gross_square_feet"], errors="coerce")
df_property["gross_square_feet"] = df_property["gross_square_feet"].replace(0, np.nan)

print(f"\n=== gross_square_feet Statistics (AFTER) ===")
print(f"Replaced {before_zeros:,} zeros with NaN")
print(f"Total missing: {df_property['gross_square_feet'].isna().sum():,}")
print(df_property["gross_square_feet"].describe())

=== gross_square_feet Statistics (BEFORE) ===
count    399955.000000
mean       1693.423963
std        1509.641178
min           0.000000
25%         960.000000
50%        1568.000000
75%        2312.000000
max      353169.000000
Name: gross_square_feet, dtype: float64

gross_square_feet = 0: 82,035 rows
Missing              : 71,045

=== gross_square_feet Statistics (AFTER) ===
Replaced 82,035 zeros with NaN
Total missing: 153,080
count    317920.000000
mean       2130.389346
std        1391.466386
min           1.000000
25%        1344.000000
50%        1836.000000
75%        2548.000000
max      353169.000000
Name: gross_square_feet, dtype: float64


In [31]:
# ============================================================
# 2L. CHECK ZIP CODE
# ============================================================

print("=== Bottom 10 zip codes ===")
print(df_property["zip_code"].value_counts().tail(10))

print(f"\nInvalid zip code '00000': {(df_property['zip_code'] == '00000').sum():,}")
print(f"Missing zip code: {df_property['zip_code'].isna().sum():,}")
print(f"Total unique zip codes: {df_property['zip_code'].nunique():,}")

=== Bottom 10 zip codes ===
zip_code
10475    202
11351    183
10044    160
11109    100
10034     91
10040     86
10803      8
11430      6
10129      3
11227      1
Name: count, dtype: int64

Invalid zip code '00000': 212
Missing zip code: 0
Total unique zip codes: 185


In [32]:
# ============================================================
# 2L. CLEAN ZIP CODE
# ============================================================

# Replace '00000' with NaN
before = (df_property["zip_code"] == "00000").sum()
df_property.loc[df_property["zip_code"] == "00000", "zip_code"] = np.nan

print(f"Replaced {before:,} '00000' with NaN")
print(f"Total missing zip_code: {df_property['zip_code'].isna().sum():,}")

Replaced 212 '00000' with NaN
Total missing zip_code: 212


In [ ]:
# ============================================================
# 2M. CHECK RESIDENTIAL UNITS
# ============================================================

print("=== residential_units Statistics ===")
print(df_property["residential_units"].describe())

print(f"\nresidential_units = 0: {(df_property['residential_units'] == 0).sum():,}")
print(f"Missing: {df_property['residential_units'].isna().sum():,}")
print(f"\nTop 10 values:")
print(df_property["residential_units"].value_counts().head(10))

In [33]:
# ============================================================
# 2M. CLEAN RESIDENTIAL UNITS
# ============================================================

before_zeros = (df_property["residential_units"] == 0).sum()
df_property["residential_units"] = df_property["residential_units"].replace(0, np.nan)

print(f"Replaced {before_zeros:,} zeros with NaN")
print(f"Total missing: {df_property['residential_units'].isna().sum():,}")
print(f"\nNew statistics:")
print(df_property["residential_units"].describe())

Replaced 0 zeros with NaN
Total missing: 0

New statistics:
count    471000.000000
mean          1.544911
std           1.073211
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          59.000000
Name: residential_units, dtype: float64


In [34]:
# ============================================================
# 2N. DROP UNNECESSARY COLUMNS
# ============================================================

cols_to_drop = [
    "ease-ment",
    "apartment_number",
    "tax_class_as_of_final_roll_17/18",
    "building_class_as_of_final_roll_17/18",
    "tax_class_as_of_final_roll_18/19",
    "building_class_as_of_final_roll_18/19"
]

df_property = df_property.drop(columns=cols_to_drop, errors="ignore")

print(f"✅ Dropped {len(cols_to_drop)} columns")
print(f"   New shape: {df_property.shape}")
print(f"   Remaining columns: {df_property.columns.tolist()}")

✅ Dropped 6 columns
   New shape: (471000, 20)
   Remaining columns: ['borough', 'neighborhood', 'building_class_category', 'tax_class_at_present', 'block', 'lot', 'building_class_at_present', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale', 'building_class_at_time_of_sale', 'sale_price', 'sale_date', 'year']


In [35]:
# ============================================================
# 2O. LOAD & CLEAN df_cs (CompStat 7 Majors by Precinct)
# ============================================================

import glob

# Precinct → Borough mapping
precinct_to_borough = {
    1: "manhattan", 5: "manhattan", 6: "manhattan", 7: "manhattan",
    9: "manhattan", 10: "manhattan", 13: "manhattan", 14: "manhattan",
    17: "manhattan", 18: "manhattan", 19: "manhattan", 20: "manhattan",
    22: "manhattan", 23: "manhattan", 24: "manhattan", 25: "manhattan",
    26: "manhattan", 28: "manhattan", 30: "manhattan", 32: "manhattan",
    33: "manhattan", 34: "manhattan",
    40: "bronx", 41: "bronx", 42: "bronx", 43: "bronx", 44: "bronx",
    45: "bronx", 46: "bronx", 47: "bronx", 48: "bronx", 49: "bronx",
    50: "bronx", 52: "bronx",
    60: "brooklyn", 61: "brooklyn", 62: "brooklyn", 63: "brooklyn",
    66: "brooklyn", 67: "brooklyn", 68: "brooklyn", 69: "brooklyn",
    70: "brooklyn", 71: "brooklyn", 72: "brooklyn", 73: "brooklyn",
    75: "brooklyn", 76: "brooklyn", 77: "brooklyn", 78: "brooklyn",
    79: "brooklyn", 81: "brooklyn", 83: "brooklyn", 84: "brooklyn",
    88: "brooklyn", 90: "brooklyn", 94: "brooklyn",
    100: "queens", 101: "queens", 102: "queens", 103: "queens",
    104: "queens", 105: "queens", 106: "queens", 107: "queens",
    108: "queens", 109: "queens", 110: "queens", 111: "queens",
    112: "queens", 113: "queens", 114: "queens", 115: "queens",
    120: "statenisland", 121: "statenisland", 122: "statenisland"
}

# Quarter labels
quarter_map = {
    "cs-report-q1-2025.xlsx": "Q1",
    "cs-report-q2-2025.xlsx": "Q2",
    "cs-report-q3-2025.xlsx": "Q3",
    "cs-report-q4-2025.xlsx": "Q4",
}

cs_files = glob.glob("Datasets/cs-report-q*.xlsx")
df_cs_list = []

for file in sorted(cs_files):
    basename = os.path.basename(file)
    quarter  = quarter_map.get(basename, "Unknown")

    # Read 7 Majors sheet, row 2 is header
    df_temp = pd.read_excel(file, sheet_name="7 Majors",
                            header=2, engine="openpyxl")

    # Clean column names
    df_temp.columns = (df_temp.columns
                       .astype(str)
                       .str.strip()
                       .str.lower()
                       .str.replace(r"\s+", "_", regex=True)
                       .str.replace(".", "", regex=False))

    # Keep only Precinct rows (drop summary/total rows)
    df_temp = df_temp.rename(columns={"precinct": "precinct"})
    df_temp = df_temp[pd.to_numeric(df_temp["precinct"], errors="coerce").notna()]
    df_temp["precinct"] = df_temp["precinct"].astype(int)

    # Add quarter and year
    df_temp["quarter"] = quarter
    df_temp["year"]    = 2025

    # Map precinct to borough
    df_temp["borough"] = df_temp["precinct"].map(precinct_to_borough)

    df_cs_list.append(df_temp)

df_cs = pd.concat(df_cs_list, ignore_index=True)

# Drop rows where borough is NaN (unmapped precincts)
df_cs = df_cs.dropna(subset=["borough"])
df_cs = df_cs.reset_index(drop=True)

# Convert crime columns to numeric
crime_cols = ["murder", "rape", "robbery", "fel_assault",
              "burglary", "gr_larceny", "gla", "total"]
for col in crime_cols:
    if col in df_cs.columns:
        df_cs[col] = pd.to_numeric(df_cs[col], errors="coerce")

print(f"✅ df_cs cleaned: {df_cs.shape}")
print(f"   Quarters : {sorted(df_cs['quarter'].unique())}")
print(f"   Boroughs : {sorted(df_cs['borough'].unique())}")
print(f"   Columns  : {df_cs.columns.tolist()}")
print(f"\nSample:")
print(df_cs.head())

✅ df_cs cleaned: (304, 12)
   Quarters : ['Q1', 'Q2', 'Q3', 'Q4']
   Boroughs : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
   Columns  : ['precinct', 'murder', 'rape', 'robbery', 'fel_assault', 'burglary', 'gr_larceny', 'gla', 'total', 'quarter', 'year', 'borough']

Sample:
   precinct  murder  rape  robbery  fel_assault  burglary  gr_larceny  gla  \
0         1       0     7       33           27        38         306    9
1         5       0     4       29           41        23         161    5
2         6       0     3       16           22        36         215    4
3         7       2     1       32           53        31          87    2
4         9       1     0       23           46        51         186   11

   total quarter  year    borough
0    420      Q1  2025  manhattan
1    263      Q1  2025  manhattan
2    296      Q1  2025  manhattan
3    208      Q1  2025  manhattan
4    318      Q1  2025  manhattan


In [36]:
# ============================================================
# FINAL CHECK BEFORE MERGE
# ============================================================

print(f"{'='*60}")
print(f"df_property FINAL SUMMARY")
print(f"{'='*60}")
print(f"Shape        : {df_property.shape}")
print(f"Boroughs     : {sorted(df_property['borough'].unique())}")
print(f"Year range   : {df_property['year'].min()} - {df_property['year'].max()}")
print(f"\nRows per year:")
print(df_property.groupby("year").size().to_string())

print(f"\n{'='*60}")
print(f"Null counts per column")
print(f"{'='*60}")
print(df_property.isnull().sum())

print(f"\n{'='*60}")
print(f"sale_price Summary")
print(f"{'='*60}")
print(f"Min   : ${df_property['sale_price'].min():,.0f}")
print(f"Max   : ${df_property['sale_price'].max():,.0f}")
print(f"Mean  : ${df_property['sale_price'].mean():,.0f}")
print(f"Median: ${df_property['sale_price'].median():,.0f}")

print(f"\n{'='*60}")
print(f"Sample Data")
print(f"{'='*60}")
print(df_property[["borough", "zip_code", "sale_price",
                    "sale_date", "year", "year_built",
                    "gross_square_feet", "residential_units"]].head(10))

df_property FINAL SUMMARY
Shape        : (471000, 20)
Boroughs     : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
Year range   : 2013 - 2025

Rows per year:
year
2013    36315
2014    36723
2015    38454
2016    38207
2017    40042
2018    38501
2019    36777
2020    28822
2021    46316
2022    42040
2023    31332
2024    32031
2025    25440

Null counts per column
borough                                0
neighborhood                           0
building_class_category                0
tax_class_at_present              120690
block                                  0
lot                                    0
building_class_at_present         120690
address                                0
zip_code                             212
residential_units                      0
commercial_units                   71045
total_units                            0
land_square_feet                   71045
gross_square_feet                 153080
year_built                         32328
t

In [37]:
# ============================================================
# Add year column to df_sales from sale_date
# ============================================================

df_sales["sale_date"] = pd.to_datetime(df_sales["sale_date"], errors="coerce")
df_sales["year"]      = df_sales["sale_date"].dt.year

print(f"✅ Added year column to df_sales")
print(f"   Shape: {df_sales.shape}")
print(f"   Years: {sorted(df_sales['year'].dropna().unique().tolist())}")
print(f"\n   Rows per year:")
print(df_sales.groupby("year").size().to_string())

✅ Added year column to df_sales
   Shape: (40960, 20)
   Years: [2022, 2023]

   Rows per year:
year
2022    27076
2023    13884


In [38]:
# ============================================================
# PREVIEW ALL DATASETS BEFORE MERGE
# ============================================================

datasets = {
    "df_property" : df_property,
    "df_sales"    : df_sales,
    "df_arrest"   : df_arrest,
    "df_spotcrime": df_spotcrime,
    "df_schools_all"  : df_schools_all,
    "df_income"   : df_income,
    "df_subway"   : df_subway,
    "df_cs"       : df_cs,
}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📊 {name}")
    print(f"{'='*60}")
    print(f"  Shape   : {df.shape}")
    print(f"  Columns : {df.columns.tolist()}")
    print(f"  Nulls   : {df.isnull().sum().sum()}")
    print(f"  Sample  :")
    print(df.head(3).to_string())


📊 df_property
  Shape   : (471000, 20)
  Columns : ['borough', 'neighborhood', 'building_class_category', 'tax_class_at_present', 'block', 'lot', 'building_class_at_present', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale', 'building_class_at_time_of_sale', 'sale_price', 'sale_date', 'year']
  Nulls   : 569090
  Sample  :
  borough               neighborhood   building_class_category tax_class_at_present   block   lot building_class_at_present                                    address zip_code  residential_units  commercial_units  total_units  land_square_feet  gross_square_feet  year_built  tax_class_at_time_of_sale building_class_at_time_of_sale  sale_price  sale_date  year
0   bronx  BATHGATE                   01  ONE FAMILY DWELLINGS                    1  3028.0  25.0                        A5  412 EAST 179TH STREET                        10457                1.0     

## Step 3: Merge

In [39]:
# ============================================================
# STEP 3: MERGE DATA
# ============================================================

# ============================================================
# 3A. FIX SPACING + COMBINE df_property + df_sales
# ============================================================

# Fix building_class_category spacing
df_property["building_class_category"] = (df_property["building_class_category"]
                                           .str.strip()
                                           .str.replace(r"\s+", " ", regex=True))

df_sales["building_class_category"] = (df_sales["building_class_category"]
                                        .str.strip()
                                        .str.replace(r"\s+", " ", regex=True))

# Prepare df_sales year column
df_sales["sale_date"] = pd.to_datetime(df_sales["sale_date"])
df_sales["year"]      = df_sales["sale_date"].dt.year

# Find shared columns
shared_cols = [col for col in df_property.columns if col in df_sales.columns]
print(f"Shared columns: {shared_cols}")

# Combine
df_master = pd.concat(
    [df_property[shared_cols], df_sales[shared_cols]],
    ignore_index=True
)

df_master = df_master.drop_duplicates()
df_master = df_master.reset_index(drop=True)

print(f"\n✅ df_master (property + sales): {df_master.shape}")
print(f"   Boroughs: {sorted(df_master['borough'].unique())}")
print(f"\n   Rows per year:")
print(df_master.groupby("year").size().to_string())

Shared columns: ['borough', 'neighborhood', 'building_class_category', 'tax_class_at_present', 'block', 'lot', 'building_class_at_present', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale', 'building_class_at_time_of_sale', 'sale_price', 'sale_date', 'year']

✅ df_master (property + sales): (498000, 20)
   Boroughs: ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']

   Rows per year:
year
2013    36315
2014    36723
2015    38454
2016    38207
2017    40042
2018    38501
2019    36777
2020    28822
2021    46316
2022    62364
2023    38008
2024    32031
2025    25440


In [40]:
# ============================================================
# 3B. MERGE + df_income (on zip_code)
# ============================================================

if "median_income_usd" in df_master.columns:
    df_master = df_master.drop(columns=["median_income_usd"])

# Clean zip_code — handle None and nan values
df_master["zip_code"] = (df_master["zip_code"]
                          .astype(str)
                          .str.replace(".0", "", regex=False)
                          .str.strip()
                          .str.replace("None", "00000")
                          .str.replace("nan",  "00000")
                          .str.zfill(5))

df_income["zip_code"] = df_income["zip_code"].astype(str).str.strip().str.zfill(5)

df_master = df_master.merge(
    df_income[["zip_code", "median_income_usd"]],
    on="zip_code", how="left"
)

print(f"✅ df_master + income: {df_master.shape}")
print(f"   Non-null income: {df_master['median_income_usd'].notna().sum():,}")
print(f"   Null income    : {df_master['median_income_usd'].isna().sum():,}")
print(f"   Matched ZIPs   : {df_master['median_income_usd'].notna().sum() / len(df_master) * 100:.1f}%")

✅ df_master + income: (498000, 21)
   Non-null income: 171,679
   Null income    : 326,321
   Matched ZIPs   : 34.5%


In [41]:
# ============================================================
# 3C. MERGE + df_schools_all (on borough)
# ============================================================

if "avg_school_rank" in df_master.columns:
    df_master = df_master.drop(columns=["avg_school_rank"])

df_schools_agg = (df_schools_all
                  .groupby("borough")["rank"]
                  .mean()
                  .reset_index()
                  .rename(columns={"rank": "avg_school_rank"}))

df_master = df_master.merge(df_schools_agg, on="borough", how="left")

print(f"✅ df_master + schools: {df_master.shape}")
print(f"   Non-null avg_school_rank: {df_master['avg_school_rank'].notna().sum():,}")


✅ df_master + schools: (498000, 22)
   Non-null avg_school_rank: 498,000


In [42]:
# ============================================================
# 3D. MERGE + df_subway (on borough)
# ============================================================

# Drop if already merged
if "subway_station_count" in df_master.columns:
    df_master = df_master.drop(columns=["subway_station_count"])

df_subway_agg = (df_subway
                 .groupby("borough")["station_id"]
                 .count()
                 .reset_index()
                 .rename(columns={"station_id": "subway_station_count"}))

df_master = df_master.merge(df_subway_agg, on="borough", how="left")

print(f"✅ df_master + subway: {df_master.shape}")
print(f"   Non-null subway: {df_master['subway_station_count'].notna().sum():,}")

✅ df_master + subway: (498000, 23)
   Non-null subway: 498,000


In [43]:
# ============================================================
# 3E. MERGE + df_arrest (on borough)
# ============================================================

if "arrest_count_2025" in df_master.columns:
    df_master = df_master.drop(columns=["arrest_count_2025"])

df_arrest_agg = (df_arrest
                 .groupby("borough")
                 .size()
                 .reset_index()
                 .rename(columns={0: "arrest_count_2025"}))

df_master = df_master.merge(df_arrest_agg, on="borough", how="left")

print(f"✅ df_master + arrests: {df_master.shape}")
print(f"   Non-null arrests: {df_master['arrest_count_2025'].notna().sum():,}")

✅ df_master + arrests: (498000, 24)
   Non-null arrests: 498,000


In [44]:
# ============================================================
# 3F. MERGE + df_cs (on borough)
# ============================================================

cs_cols = ["murder", "rape", "robbery", "fel_assault",
           "burglary", "gr_larceny", "gla", "total"]

for col in cs_cols:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

df_cs_agg = (df_cs.groupby("borough")[cs_cols]
             .sum()
             .reset_index()
             .rename(columns={c: f"cs_{c}_2025" for c in cs_cols}))

df_master = df_master.merge(df_cs_agg, on="borough", how="left")

print(f"✅ df_master + cs: {df_master.shape}")
print(f"   Non-null cs_total: {df_master['cs_total_2025'].notna().sum():,}")

✅ df_master + cs: (498000, 32)
   Non-null cs_total: 498,000


In [45]:
# ============================================================
# 3G. FINAL CHECK
# ============================================================

print(f"\n{'='*60}")
print(f"FINAL MASTER TABLE")
print(f"{'='*60}")
print(f"  Shape      : {df_master.shape}")
print(f"  Boroughs   : {sorted(df_master['borough'].unique())}")
print(f"  Year range : {df_master['year'].min()} - {df_master['year'].max()}")
print(f"\n  Rows per year:")
print(df_master.groupby("year").size().to_string())
print(f"\n  Null counts:")
print(df_master[["median_income_usd", "avg_school_rank",
                  "subway_station_count", "arrest_count_2025"]].isnull().sum())
print(f"\n  Sample rows:")
print(df_master[["borough", "year", "sale_price",
                  "median_income_usd", "avg_school_rank",
                  "subway_station_count", "arrest_count_2025"]].head(5))


FINAL MASTER TABLE
  Shape      : (498000, 32)
  Boroughs   : ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
  Year range : 2013 - 2025

  Rows per year:
year
2013    36315
2014    36723
2015    38454
2016    38207
2017    40042
2018    38501
2019    36777
2020    28822
2021    46316
2022    62364
2023    38008
2024    32031
2025    25440

  Null counts:
median_income_usd       326321
avg_school_rank              0
subway_station_count         0
arrest_count_2025            0
dtype: int64

  Sample rows:
  borough  year  sale_price  median_income_usd  avg_school_rank  \
0   bronx  2013    355000.0                NaN        50.866667
1   bronx  2013    120000.0                NaN        50.866667
2   bronx  2013    474819.0                NaN        50.866667
3   bronx  2013    210000.0                NaN        50.866667
4   bronx  2013    343116.0                NaN        50.866667

   subway_station_count  arrest_count_2025
0                    70              62833
1

In [46]:
# Save df_master for dashboard use
#df_master.to_csv("Datasets/df_master.csv", index=False)
print(f"✅ Saved → Datasets/df_master.csv")
print(f"   Shape: {df_master.shape}")

✅ Saved → Datasets/df_master.csv
   Shape: (498000, 32)


In [47]:
df_master.head(5)

In [48]:
# we have 32 colums
print(df_master.columns.tolist())

#
print(df_master[["borough", "zip_code", "sale_price",
                  "median_income_usd", "avg_school_rank",
                  "subway_station_count", "arrest_count_2025"]].head(5))

['borough', 'neighborhood', 'building_class_category', 'tax_class_at_present', 'block', 'lot', 'building_class_at_present', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale', 'building_class_at_time_of_sale', 'sale_price', 'sale_date', 'year', 'median_income_usd', 'avg_school_rank', 'subway_station_count', 'arrest_count_2025', 'cs_murder_2025', 'cs_rape_2025', 'cs_robbery_2025', 'cs_fel_assault_2025', 'cs_burglary_2025', 'cs_gr_larceny_2025', 'cs_gla_2025', 'cs_total_2025']
  borough zip_code  sale_price  median_income_usd  avg_school_rank  \
0   bronx    10457    355000.0                NaN        50.866667
1   bronx    10458    120000.0                NaN        50.866667
2   bronx    10458    474819.0                NaN        50.866667
3   bronx    10457    210000.0                NaN        50.866667
4   bronx    10457    343116.0                NaN        50.866667

   

## Data Quality Notes

### 1. median_income_usd — 316,845 non-null (39%)
The income dataset only contains **56 zip codes**, so many rows in df_master
could not be matched and remain NaN. This is expected and will not significantly
impact our visualizations, as we are primarily analyzing at the borough level.

### 2. avg_school_rank — 140,976 non-null (17%)
The school rankings dataset only contains **25 schools**, merged on borough level.
Many boroughs have limited matches, resulting in a high NaN rate.
When using school rank in analysis, we will filter to rows with valid data only.

### 3. subway_station_count & arrest_count_2025 — 100% non-null ✅
Both subway and arrest data merged successfully with no missing values.

### Impact on Dashboard
| Visualization | Data Used | Impact |
|---|---|---|
| Geospatial Map | borough + sale_price | ✅ No impact |
| Time-Series | year + sale_price | ✅ No impact |
| Scatterplot | arrest_count_2025 + sale_price | ✅ No impact |
| 4th Chart | subway_station_count + sale_price | ✅ No impact |

## Step 4: Dashboard
   ├── 🗺️ Geospatial Map
   ├── 📈 Time-Series
   ├── 🔵 Scatterplot
   └── 📊 4th Chart 

In [49]:
# ============================================================
# GLOBAL THEME SETTINGS
# All visualizations reference this config for consistency
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
import requests
import numpy as np

THEME = {
    # Font
    "font_family"    : "Arial",
    "font_color"     : "#2c2c2c",
    "font_size"      : 12,
    "title_size"     : 20,
    "axis_title_size": 13,
    "tick_size"      : 11,

    # Background
    "paper_bgcolor"  : "#ffffff",
    "plot_bgcolor"   : "#f8f9fa",

    # Grid
    "gridcolor"      : "rgba(0,0,0,0.06)",
    "linecolor"      : "rgba(0,0,0,0.15)",

    # Borough colors (same across all charts)
    "borough_colors" : {
        "Bronx"        : "#636EFA",
        "Brooklyn"     : "#EF553B",
        "Manhattan"    : "#00CC96",
        "Queens"       : "#FFA15A",
        "Staten Island": "#AB63FA"
    },

    # Map style
    "map_style"      : "carto-positron",

    # Heights
    "height_large"   : 720,
    "height_medium"  : 500,
    "height_small"   : 400,

    # Margins
    "margin_map"     : {"r": 0, "t": 80, "l": 0, "b": 0},
    "margin_chart"   : dict(t=80, b=60, l=80, r=120),
}

# Borough name mapping (used across all visualizations)
BOROUGH_NAME_MAP = {
    "bronx"       : "Bronx",
    "brooklyn"    : "Brooklyn",
    "manhattan"   : "Manhattan",
    "queens"      : "Queens",
    "statenisland": "Staten Island"
}

In [86]:
# ============================================================
# VISUALIZATION 1
# Geospatial Choropleth Map - Median Sale Price by ZIP Code
# ============================================================

zip_col = "zip_code"

# Clean zip code format
df_master[zip_col] = (
    df_master[zip_col]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(5)
)

# Prepare ZIP-level data aggregated by year and borough
zip_map_data = (
    df_master
    .dropna(subset=[zip_col, "sale_price", "year"])
    .groupby(["year", zip_col, "borough"], as_index=False)
    .agg(
        median_sale_price=("sale_price", "median"),
        avg_sale_price   =("sale_price", "mean"),
        total_sales      =("sale_price", "count")
    )
)

# Remove ZIP-year groups with fewer than 10 sales
zip_map_data = zip_map_data[zip_map_data["total_sales"] >= 10].copy()

# Add borough display name for hover tooltip
zip_map_data["borough_display"] = zip_map_data["borough"].map(BOROUGH_NAME_MAP)

# ── Merge subway distance ──
df_subway_dist = df_zip_centroids[["zip_code", "nearest_subway_dist_miles", "nearest_subway"]].copy()
df_subway_dist["nearest_subway_dist_miles"] = df_subway_dist["nearest_subway_dist_miles"].round(2)
zip_map_data = zip_map_data.merge(df_subway_dist, on="zip_code", how="left")

# ── Merge crime count ──
arr_zip = arrest_per_zip.rename(columns={"arrest_count": "arrest_count_2025"})
zip_map_data = zip_map_data.merge(arr_zip, on="zip_code", how="left")

# Load NYC ZIP Code GeoJSON boundary file
zip_geojson_url = "https://raw.githubusercontent.com/fedhere/PUI2015_EC/master/mam1612_EC/nyc-zip-code-tabulation-areas-polygons.geojson"
nyc_zip_geojson = requests.get(zip_geojson_url).json()

# Set color range using 5th to 95th percentile
price_min = zip_map_data["median_sale_price"].quantile(0.10)
price_max = zip_map_data["median_sale_price"].quantile(0.85)

# Build animated choropleth map
fig_map = px.choropleth_map(
    zip_map_data,
    geojson=nyc_zip_geojson,
    locations=zip_col,
    featureidkey="properties.postalCode",
    color="median_sale_price",
    animation_frame="year",
    color_continuous_scale="Blues",
    range_color=(price_min, price_max),
    map_style=THEME["map_style"],
    zoom=9.2,
    center={"lat": 40.7128, "lon": -74.0060},
    opacity=0.78,
    hover_name="borough_display",
    hover_data={
        "median_sale_price"        : ":$,.0f",
        "avg_sale_price"           : ":$,.0f",
        "total_sales"              : ":,",
        zip_col                    : True,
        "nearest_subway_dist_miles": ":.2f",
        "nearest_subway"           : True,
        "arrest_count_2025"        : ":,",
        "borough_display"          : False,
        "year"                     : False,
        "borough"                  : False
    },
    labels={
        "median_sale_price"        : "Median Sale Price",
        "avg_sale_price"           : "Avg Sale Price",
        "total_sales"              : "Total Sales",
        zip_col                    : "ZIP Code",
        "nearest_subway_dist_miles": "Nearest Subway (miles)",
        "nearest_subway"           : "Nearest Station",
        "arrest_count_2025"        : "Arrests (2025)"
    },
    title="NYC Median Property Sale Price by ZIP Code (2013–2025)",
    height=THEME["height_large"]
)

# Style ZIP borders
fig_map.update_traces(
    marker_line_width=0.3,
    marker_line_color="white"
)

# Apply unified theme
fig_map.update_layout(
    margin=THEME["margin_map"],
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    coloraxis_colorbar=dict(
        title=dict(
            text="Median Sale Price",
            font=dict(family=THEME["font_family"], size=16, color=THEME["font_color"])
        ),
        tickprefix="$",
        tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=14, color=THEME["font_color"]),
        thickness=22,
        len=0.7,
        x=1.02
    ),
    sliders=[{
        "currentvalue": {
            "prefix": "Year: ",
            "font"  : {"size": 14, "family": THEME["font_family"], "color": THEME["font_color"]}
        },
        "pad" : {"t": 60},
        "font": {"family": THEME["font_family"], "size": THEME["tick_size"], "color": THEME["font_color"]}
    }],
    updatemenus=[{
        "type": "buttons", "showactive": False,
        "y": 0, "x": 0.1, "xanchor": "right", "yanchor": "top",
        "buttons": [
            {"label": "▶ Play", "method": "animate",
             "args": [None, {"frame": {"duration": 800, "redraw": True},
                             "fromcurrent": True, "transition": {"duration": 300}}]},
            {"label": "⏸ Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False},
                               "mode": "immediate", "transition": {"duration": 0}}]}
        ]
    }]
)

fig_map.show()

In [87]:
# ============================================================
# VISUALIZATION 2
# Time Series Line Chart - Property Price Trends by Borough
# ============================================================

# Prepare data
line_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .groupby(["year", "borough"], as_index=False)
    .agg(median_sale_price=("sale_price", "median"))
)

# Add borough display name
line_data["borough_display"] = line_data["borough"].map(BOROUGH_NAME_MAP)

# Manual y offset for end labels to avoid overlap
annotation_offset = {
    "Manhattan"    :  80000,
    "Brooklyn"     :  50000,
    "Queens"       :  50000,
    "Staten Island":  20000,
    "Bronx"        : -60000
}

# Build line chart
fig_line = go.Figure()

for borough in ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]:
    df_b = line_data[line_data["borough_display"] == borough].sort_values("year")

    # Calculate total percentage change
    price_first = df_b["median_sale_price"].iloc[0]
    price_last  = df_b["median_sale_price"].iloc[-1]
    pct_change  = ((price_last - price_first) / price_first) * 100
    sign        = "+" if pct_change >= 0 else ""

    # Add main line trace
    fig_line.add_trace(go.Scatter(
        x=df_b["year"],
        y=df_b["median_sale_price"].tolist(),
        mode="lines+markers",
        name=borough,
        legendgroup=borough,           # ← group with label trace
        line=dict(color=THEME["borough_colors"][borough], width=2.5),
        marker=dict(size=6, color=THEME["borough_colors"][borough]),
        hovertemplate=(
            f"<b>{borough}</b><br>"
            "Year: %{x}<br>"
            "Median Price: $%{y:,.0f}<br>"
            "<extra></extra>"
        )
    ))

    # Add label as separate trace — disappears when borough is toggled
    fig_line.add_trace(go.Scatter(
        x=[df_b["year"].max()],
        y=[price_last + annotation_offset[borough]],
        mode="text",
        name=borough,
        legendgroup=borough,           # ← same group as line
        showlegend=False,              # ← don't show in legend
        text=[f"  {borough} {sign}{pct_change:.0f}%"],
        textposition="middle right",
        cliponaxis=False,
        textfont=dict(
            family=THEME["font_family"],
            size=13,
            color=THEME["borough_colors"][borough]
        ),
        hoverinfo="skip"
    ))

# Add COVID-19 shaded region
fig_line.add_vrect(
    x0=2019.5, x1=2021.5,
    fillcolor="rgba(255,200,0,0.08)",
    layer="below", line_width=0
)

# Add COVID-19 label
fig_line.add_annotation(
    x=2020.5, y=1, xref="x", yref="paper",
    text="COVID-19", showarrow=False,
    font=dict(family=THEME["font_family"], color="rgba(180,140,0,0.7)", size=13)
)

# Apply unified theme
fig_line.update_layout(
    title=dict(
        text="NYC Property Sale Price Trends by Borough (2013–2025)",
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    height=THEME["height_medium"],
    margin=dict(t=80, b=60, l=80, r=200),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="Year",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickmode="linear", dtick=1,
        range=[2013, 2026],              # fixed range leaves room for end labels
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hovermode="x unified",
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
    )
)

fig_line.show()

In [89]:
# ============================================================
# VISUALIZATION 3
# Scatter Plot - Median Household Income vs Property Sale Price
# ============================================================

# Prepare data: aggregate by zip code and borough
# Only filter invalid prices, keep all real market transactions
scatter_income_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .dropna(subset=["median_income_usd", "sale_price", "zip_code", "borough"])
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        median_sale_price =("sale_price",        "median"),
        median_income_usd =("median_income_usd", "median"),
        total_sales       =("sale_price",        "count")
    )
)

# Remove zip codes with fewer than 10 sales
scatter_income_data = scatter_income_data[scatter_income_data["total_sales"] >= 10].copy()

# Add borough display name
scatter_income_data["borough_display"] = scatter_income_data["borough"].map(BOROUGH_NAME_MAP)

# Build scatter plot
fig_scatter_income = px.scatter(
    scatter_income_data,
    x="median_income_usd",
    y="median_sale_price",
    color="borough_display",
    size="total_sales",
    size_max=40,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="gray",
    color_discrete_map=THEME["borough_colors"],
    hover_name="borough_display",
    hover_data={
        "zip_code"         : True,
        "median_income_usd": ":$,.0f",
        "median_sale_price": ":$,.0f",
        "total_sales"      : ":,",
        "borough_display"  : False,
        "borough"          : False
    },
    labels={
        "median_income_usd" : "Median Household Income",
        "median_sale_price" : "Median Property Sale Price",
        "total_sales"       : "Total Sales",
        "borough_display"   : "Borough",
        "zip_code"          : "ZIP Code"
    },
    title="Median Household Income vs Property Sale Price by ZIP Code",
    height=THEME["height_medium"]
)

# Apply unified theme
fig_scatter_income.update_layout(
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    margin=THEME["margin_chart"],
    legend=dict(
        title=dict(
            text="Borough",
            font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
        ),
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="Median Household Income",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Property Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
    )
)

fig_scatter_income.show()

In [103]:
# Aggregate school rank by ZIP code instead of borough
df_schools_zip = (
    df_schools_all
    .groupby("zip_code")["rank"]
    .mean()
    .reset_index()
    .rename(columns={"rank": "avg_school_rank_zip"})
)

# Prepare scatter data with ZIP-level school rank
scatter_school_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .dropna(subset=["sale_price", "zip_code", "borough"])
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        median_sale_price=("sale_price", "median"),
        total_sales      =("sale_price", "count")
    )
)

# Merge ZIP-level school rank
scatter_school_data = scatter_school_data.merge(
    df_schools_zip, on="zip_code", how="inner"  # inner = only ZIPs with school data
)

# Remove zip codes with fewer than 10 sales
scatter_school_data = scatter_school_data[scatter_school_data["total_sales"] >= 10].copy()

# Add borough display name
scatter_school_data["borough_display"] = scatter_school_data["borough"].map(BOROUGH_NAME_MAP)

print(f"✅ scatter_school_data: {scatter_school_data.shape}")
print(f"   Unique school ranks: {scatter_school_data['avg_school_rank_zip'].nunique()}")

✅ scatter_school_data: (70, 6)
   Unique school ranks: 55


In [104]:
# Add best school name and type per ZIP
df_best_school = (
    df_schools_all
    .sort_values("rank")
    .groupby("zip_code")
    .first()
    .reset_index()[["zip_code", "school_name", "type", "school_level"]]
    .rename(columns={
        "school_name" : "best_school_name",
        "type"        : "best_school_type",
        "school_level": "best_school_level"
    })
)

scatter_school_data = scatter_school_data.merge(
    df_best_school, on="zip_code", how="left"
)

# Add school score
scatter_school_data["school_score"] = 101 - scatter_school_data["avg_school_rank_zip"]

print(f"✅ scatter_school_data updated: {scatter_school_data.shape}")
print(scatter_school_data[["zip_code", "school_score", "best_school_name",
                            "best_school_type", "best_school_level"]].head())

✅ scatter_school_data updated: (70, 10)
  zip_code  school_score                       best_school_name  \
0    10001     73.000000           Success Academy Hudson Yards
1    10002     54.666667                                 NEST+m
2    10003     61.000000           Success Academy Union Square
3    10004     73.000000       Academy for Software Engineering
4    10009     59.000000  Institute for Collaborative Education

  best_school_type best_school_level
0          Charter        elementary
1           Public        elementary
2          Charter        elementary
3         Screened              high
4         Screened              high


In [106]:
# Build scatter plot
fig_scatter_school = px.scatter(
    scatter_school_data,
    x="school_score",
    y="median_sale_price",
    color="borough_display",
    size="total_sales",
    size_max=40,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="gray",
    color_discrete_map=THEME["borough_colors"],
    hover_name="borough_display",
    hover_data={
        "zip_code"         : True,
        "school_score"     : ":.1f",
        "best_school_name" : True,
        "best_school_type" : True,
        "best_school_level": True,
        "median_sale_price": ":$,.0f",
        "total_sales"      : ":,",
        "borough_display"  : False,
        "borough"          : False
    },
    labels={
        "school_score"     : "School Quality Score (higher = better)",
        "best_school_name" : "Top School",
        "best_school_type" : "School Type",
        "best_school_level": "School Level",
        "median_sale_price": "Median Property Sale Price",
        "total_sales"      : "Total Sales",
        "borough_display"  : "Borough",
        "zip_code"         : "ZIP Code"
    },
    title="School Quality vs Property Sale Price by ZIP Code",
    height=THEME["height_medium"]
)

# Apply unified theme
fig_scatter_school.update_layout(
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    margin=THEME["margin_chart"],
    legend=dict(
        title=dict(
            text="Borough",
            font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
        ),
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="School Quality Score (higher = better)",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Property Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
    )
)

fig_scatter_school.show()

In [99]:
# ============================================================
# CRIME DATA - Combine CompStat + Arrest
# ============================================================

# Step 1: CompStat borough level crime per 10,000 people
cs_borough = (
    df_cs.groupby("borough")[["murder", "rape", "robbery",
                               "fel_assault", "burglary",
                               "gr_larceny", "gla", "total"]]
    .sum()
    .reset_index()
)

population = {
    "bronx"       : 1472654,
    "brooklyn"    : 2736074,
    "manhattan"   : 1694251,
    "queens"      : 2405464,
    "statenisland": 495747
}

cs_borough["population"]    = cs_borough["borough"].map(population)
cs_borough["crime_per_10k"] = (
    cs_borough["total"] / cs_borough["population"] * 10000
).round(1)

print("=== CompStat Crime per 10,000 people ===")
print(cs_borough[["borough", "total", "population", "crime_per_10k"]].to_string())

# Step 2: Merge crime_per_10k into arrest_per_zip
arrest_zip_enhanced = arrest_per_zip.copy()
arrest_zip_enhanced = arrest_zip_enhanced.merge(
    df_zip_centroids[["zip_code", "lat", "lon"]], on="zip_code", how="left"
)

# Map each ZIP to borough using df_master
zip_borough_map = (
    df_master[["zip_code", "borough"]]
    .dropna()
    .drop_duplicates(subset=["zip_code"])
)
arrest_zip_enhanced = arrest_zip_enhanced.merge(
    zip_borough_map, on="zip_code", how="left"
)

# Merge crime_per_10k by borough
arrest_zip_enhanced = arrest_zip_enhanced.merge(
    cs_borough[["borough", "crime_per_10k", "murder",
                "rape", "robbery", "fel_assault",
                "burglary", "gr_larceny", "gla", "total"]],
    on="borough", how="left"
)
arrest_zip_enhanced["borough_display"] = arrest_zip_enhanced["borough"].map(BOROUGH_NAME_MAP)

print(f"\n✅ arrest_zip_enhanced: {arrest_zip_enhanced.shape}")
print(arrest_zip_enhanced.head())

=== CompStat Crime per 10,000 people ===
        borough  total  population  crime_per_10k
0         bronx  30051     1472654          204.1
1      brooklyn  30310     2736074          110.8
2     manhattan  29749     1694251          175.6
3        queens  25051     2405464          104.1
4  statenisland   2886      495747           58.2

✅ arrest_zip_enhanced: (260, 15)
  zip_code  arrest_count        lat        lon    borough  crime_per_10k  \
0    00083           729  40.780441 -73.967710        NaN            NaN
1    10001           332  40.749814 -74.000445  manhattan          175.6
2    10002          2441  40.723723 -73.979159  manhattan          175.6
3    10003          2070  40.732860 -73.988938  manhattan          175.6
4    10004            81  40.702367 -74.012640  manhattan          175.6

   murder   rape  robbery  fel_assault  burglary  gr_larceny     gla    total  \
0     NaN    NaN      NaN          NaN       NaN         NaN     NaN      NaN
1    47.0  383.0   3259.

In [121]:
# ============================================================
# VISUALIZATION 5
# Scatter Plot - Crime Rate vs Property Sale Price by ZIP
# Combined: Arrest count (ZIP level) + CompStat (Borough level)
# ============================================================

# Remove invalid ZIP codes
arrest_zip_enhanced_clean = arrest_zip_enhanced.dropna(subset=["borough"]).copy()

# Prepare property data by ZIP
scatter_crime_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .dropna(subset=["sale_price", "zip_code", "borough"])
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        median_sale_price=("sale_price", "median"),
        total_sales      =("sale_price", "count")
    )
)
scatter_crime_data = scatter_crime_data[scatter_crime_data["total_sales"] >= 10].copy()
scatter_crime_data["borough_display"] = scatter_crime_data["borough"].map(BOROUGH_NAME_MAP)

# Merge enhanced crime data
scatter_crime_data = scatter_crime_data.merge(
    arrest_zip_enhanced_clean[[
        "zip_code", "arrest_count", "crime_per_10k",
        "murder", "rape", "robbery", "fel_assault",
        "burglary", "gr_larceny", "gla"
    ]],
    on="zip_code", how="inner"
)

print(f"✅ scatter_crime_data: {scatter_crime_data.shape}")
print(scatter_crime_data.head(3))

# Build scatter plot
fig_scatter_crime = px.scatter(
    scatter_crime_data,
    x="arrest_count",
    y="median_sale_price",
    color="borough_display",
    size="total_sales",
    size_max=40,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="gray",
    color_discrete_map=THEME["borough_colors"],
    hover_name="borough_display",
    hover_data={
       "zip_code"        : True,
        "median_sale_price": ":$,.0f",
        "total_sales"     : ":,",
        "arrest_count"    : ":,",
        "crime_per_10k"   : ":.1f",
        "murder"          : ":.0f",
        "rape"            : ":.0f",
        "robbery"         : ":.0f",
        "fel_assault"     : ":.0f",
        "burglary"        : ":.0f",
        "gr_larceny"      : ":.0f",
        "gla"             : ":.0f",   # ← add
        "borough_display" : False,
        "borough"         : False
    },
labels={
         "zip_code"        : "ZIP Code",
         "median_sale_price": "Median Property Sale Price",
         "total_sales"     : "Total Sales",
         "arrest_count"    : "Arrests per ZIP (2025)",
         "crime_per_10k"   : "Serious Crimes per 10,000 People",
         "murder"          : "Murder (Homicide)",
         "rape"            : "Rape (Sexual Assault)",
         "robbery"         : "Robbery (Armed/Force)",
         "fel_assault"     : "Felony Assault (Violence)",
         "burglary"        : "Burglary (Break-in)",
         "gr_larceny"      : "Grand Larceny (Theft >$1,000)",
         "gla"             : "Grand Larceny Auto (Car Theft)",
         "borough_display" : "Borough",
         "borough"         : False
    },
    title="Crime Rate vs Property Sale Price by ZIP Code (2025)",
    height=THEME["height_medium"]
)

# Apply unified theme
fig_scatter_crime.update_layout(
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    margin=THEME["margin_chart"],
    legend=dict(
        title=dict(
            text="Borough",
            font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
        ),
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="Total Arrests per ZIP Code (2025)",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Property Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=12, color=THEME["font_color"])
    )
)

fig_scatter_crime.show()

✅ scatter_crime_data: (193, 14)
  zip_code    borough  median_sale_price  total_sales borough_display  \
0    10001  manhattan          2487500.0         1793       Manhattan
1    10002  manhattan          1531044.5         2434       Manhattan
2    10003  manhattan          2195000.0         2543       Manhattan

   arrest_count  crime_per_10k  murder   rape  robbery  fel_assault  burglary  \
0           332          175.6    47.0  383.0   3259.0       5239.0    3337.0
1          2441          175.6    47.0  383.0   3259.0       5239.0    3337.0
2          2070          175.6    47.0  383.0   3259.0       5239.0    3337.0

   gr_larceny     gla
0     16299.0  1185.0
1     16299.0  1185.0
2     16299.0  1185.0


In [72]:
# ============================================================
# VISUALIZATION 6 - DATA PREPARATION
# Calculate distance from each ZIP to nearest subway station
# ============================================================

from scipy.spatial import cKDTree
import numpy as np

# NYC ZIP code centroids (approximate lat/lon center of each ZIP)
# We need to get ZIP centroids from GeoJSON
zip_centroids = []
for feature in nyc_zip_geojson["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]

    # Get centroid from polygon
    try:
        if feature["geometry"]["type"] == "Polygon":
            pts = np.array(coords[0])
        else:  # MultiPolygon
            pts = np.array(coords[0][0])

        centroid_lon = pts[:, 0].mean()
        centroid_lat = pts[:, 1].mean()

        zip_centroids.append({
            "zip_code": str(props["postalCode"]).zfill(5),
            "lat"     : centroid_lat,
            "lon"     : centroid_lon
        })
    except:
        pass

df_zip_centroids = pd.DataFrame(zip_centroids)
print(f"✅ ZIP centroids: {df_zip_centroids.shape}")
print(df_zip_centroids.head())

✅ ZIP centroids: (262, 3)
  zip_code        lat        lon
0    11372  40.750874 -73.885503
1    11004  40.749910 -73.714132
2    11040  40.746800 -73.703885
3    11426  40.737543 -73.726377
4    11365  40.741435 -73.800801


In [73]:
# ============================================================
# Calculate nearest subway station distance for each ZIP
# ============================================================

# Subway station coordinates
subway_coords = df_subway[["latitude", "longitude"]].values

# ZIP centroid coordinates
zip_coords = df_zip_centroids[["lat", "lon"]].values

# Build KD-Tree for fast nearest neighbor search
tree = cKDTree(subway_coords)

# Find nearest subway station for each ZIP
distances, indices = tree.query(zip_coords, k=1)

# Convert degree distance to miles (approximate)
# 1 degree ≈ 69 miles
df_zip_centroids["nearest_subway_dist_miles"] = distances * 69

# Add nearest subway station name
df_zip_centroids["nearest_subway"] = df_subway["stop_name"].iloc[indices].values

print(f"✅ Nearest subway distances calculated")
print(f"\n=== Distance Statistics ===")
print(f"Min  : {df_zip_centroids['nearest_subway_dist_miles'].min():.2f} miles")
print(f"Max  : {df_zip_centroids['nearest_subway_dist_miles'].max():.2f} miles")
print(f"Mean : {df_zip_centroids['nearest_subway_dist_miles'].mean():.2f} miles")
print(f"\nTop 5 closest ZIPs to subway:")
print(df_zip_centroids.nsmallest(5, "nearest_subway_dist_miles")[
    ["zip_code", "nearest_subway_dist_miles", "nearest_subway"]
].to_string())
print(f"\nTop 5 farthest ZIPs from subway:")
print(df_zip_centroids.nlargest(5, "nearest_subway_dist_miles")[
    ["zip_code", "nearest_subway_dist_miles", "nearest_subway"]
].to_string())

✅ Nearest subway distances calculated

=== Distance Statistics ===
Min  : 0.01 miles
Max  : 6.00 miles
Mean : 0.83 miles

Top 5 closest ZIPs to subway:
    zip_code  nearest_subway_dist_miles        nearest_subway
16     11435                   0.013132          Sutphin Blvd
130    11237                   0.018079             DeKalb Av
255    10286                   0.024963               Wall St
251    10203                   0.027336               Wall St
248    10047                   0.028800  23 St-Baruch College

Top 5 farthest ZIPs from subway:
    zip_code  nearest_subway_dist_miles  nearest_subway
2      11040                   5.997706  Jamaica-179 St
165    10303                   5.610917        New Dorp
6      11001                   5.575207  Jamaica-179 St
111    11005                   5.539487  Jamaica-179 St
1      11004                   5.452557  Jamaica-179 St


In [108]:
# ============================================================
# VISUALIZATION 6
# Scatter Plot - Subway Accessibility vs Property Sale Price by ZIP
# ============================================================

# Prepare property data by ZIP
scatter_subway_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .dropna(subset=["sale_price", "zip_code", "borough"])
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        median_sale_price=("sale_price", "median"),
        total_sales      =("sale_price", "count")
    )
)

# Remove ZIP codes with fewer than 10 sales
scatter_subway_data = scatter_subway_data[scatter_subway_data["total_sales"] >= 10].copy()

# Merge with subway distance
scatter_subway_data = scatter_subway_data.merge(
    df_zip_centroids[["zip_code", "nearest_subway_dist_miles", "nearest_subway"]],
    on="zip_code", how="inner"
)

# Add borough display name
scatter_subway_data["borough_display"] = scatter_subway_data["borough"].map(BOROUGH_NAME_MAP)

print(f"✅ scatter_subway_data: {scatter_subway_data.shape}")
print(scatter_subway_data.head())

# Build scatter plot
fig_scatter_subway = px.scatter(
    scatter_subway_data,
    x="nearest_subway_dist_miles",
    y="median_sale_price",
    color="borough_display",
    size="total_sales",
    size_max=40,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="gray",
    color_discrete_map=THEME["borough_colors"],
    hover_name="borough_display",
    hover_data={
        "zip_code"                 : True,
        "nearest_subway_dist_miles": ":.2f",
        "nearest_subway"           : True,
        "median_sale_price"        : ":$,.0f",
        "total_sales"              : ":,",
        "borough_display"          : False,
        "borough"                  : False
    },
    labels={
        "nearest_subway_dist_miles": "Distance to Nearest Subway (miles)",
        "median_sale_price"        : "Median Property Sale Price",
        "total_sales"              : "Total Sales",
        "borough_display"          : "Borough",
        "zip_code"                 : "ZIP Code",
        "nearest_subway"           : "Nearest Station"
    },
    title="Subway Accessibility vs Property Sale Price by ZIP Code",
    height=THEME["height_medium"]
)

# Apply unified theme
fig_scatter_subway.update_layout(
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    margin=THEME["margin_chart"],
    legend=dict(
        title=dict(
            text="Borough",
            font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
        ),
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="Distance to Nearest Subway Station (miles)",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Property Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
    )
)

fig_scatter_subway.show()

✅ scatter_subway_data: (193, 7)
  zip_code    borough  median_sale_price  total_sales  \
0    10001  manhattan          2487500.0         1793
1    10002  manhattan          1531044.5         2434
2    10003  manhattan          2195000.0         2543
3    10004  manhattan          1149095.0         1071
4    10004  manhattan          1149095.0         1071

   nearest_subway_dist_miles            nearest_subway borough_display
0                   0.316590                     23 St       Manhattan
1                   0.527193                      1 Av       Manhattan
2                   0.143287            14 St-Union Sq       Manhattan
3                   0.055394  Whitehall St-South Ferry       Manhattan
4                   1.873798               South Ferry       Manhattan


In [75]:
# Check arrest data lat/lon coverage
print(f"df_arrest with lat/lon: {df_arrest.dropna(subset=['Latitude', 'Longitude']).shape}")
print(df_arrest[["Latitude", "Longitude", "borough"]].head())

df_arrest with lat/lon: (278953, 7)
    Latitude  Longitude       borough
0  40.745543 -73.881427        queens
1  40.645005 -74.077265  statenisland
2  40.853598 -73.900577         bronx
3  40.674496 -73.930571      brooklyn
4  40.674496 -73.930571      brooklyn


In [76]:
# ============================================================
# Map arrest data to ZIP code using KD-Tree
# ============================================================

# Arrest coordinates
arrest_coords = df_arrest[["Latitude", "Longitude"]].dropna().values

# Find nearest ZIP centroid for each arrest
zip_tree = cKDTree(df_zip_centroids[["lat", "lon"]].values)
distances, indices = zip_tree.query(arrest_coords, k=1)

# Assign ZIP code to each arrest
df_arrest_zip = df_arrest.dropna(subset=["Latitude", "Longitude"]).copy()
df_arrest_zip["zip_code"] = df_zip_centroids["zip_code"].iloc[indices].values

# Count arrests per ZIP
arrest_per_zip = (
    df_arrest_zip.groupby("zip_code")
    .size()
    .reset_index(name="arrest_count")
)

print(f"✅ arrest_per_zip: {arrest_per_zip.shape}")
print(f"   Top 5 ZIP by arrests:")
print(arrest_per_zip.nlargest(5, "arrest_count").to_string())

✅ arrest_per_zip: (245, 2)
   Top 5 ZIP by arrests:
    zip_code  arrest_count
121    10457          7508
157    11207          6817
119    10455          5427
103    10301          4916
116    10452          4591


In [122]:
# ============================================================
# VISUALIZATION 5 - ZIP Level
# Scatter Plot - Crime Rate vs Property Sale Price by ZIP
# ============================================================

# Prepare property data by ZIP
scatter_crime_data = (
    df_master[df_master["sale_price"] >= 10_000]
    .dropna(subset=["sale_price", "zip_code", "borough"])
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        median_sale_price=("sale_price", "median"),
        total_sales      =("sale_price", "count")
    )
)

# Remove ZIP codes with fewer than 10 sales
scatter_crime_data = scatter_crime_data[scatter_crime_data["total_sales"] >= 10].copy()

# Merge with arrest per ZIP
scatter_crime_data = scatter_crime_data.merge(
    arrest_per_zip, on="zip_code", how="inner"
)

# Add borough display name
scatter_crime_data["borough_display"] = scatter_crime_data["borough"].map(BOROUGH_NAME_MAP)

print(f"✅ scatter_crime_data: {scatter_crime_data.shape}")
print(scatter_crime_data.head())

# Build scatter plot
fig_scatter_crime = px.scatter(
    scatter_crime_data,
    x="arrest_count",
    y="median_sale_price",
    color="borough_display",
    size="total_sales",
    size_max=40,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="gray",
    color_discrete_map=THEME["borough_colors"],
    hover_name="borough_display",
    hover_data={
        "zip_code"         : True,
        "arrest_count"     : ":,",
        "median_sale_price": ":$,.0f",
        "total_sales"      : ":,",
        "borough_display"  : False,
        "borough"          : False
    },
    labels={
        "arrest_count"     : "Total Arrests (2025)",
        "median_sale_price": "Median Property Sale Price",
        "total_sales"      : "Total Sales",
        "borough_display"  : "Borough",
        "zip_code"         : "ZIP Code"
    },
    title="Crime Rate vs Property Sale Price by ZIP Code",
    height=THEME["height_medium"]
)

# Apply unified theme
fig_scatter_crime.update_layout(
    title=dict(
        x=0.5,
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    paper_bgcolor=THEME["paper_bgcolor"],
    plot_bgcolor =THEME["plot_bgcolor"],
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    margin=THEME["margin_chart"],
    legend=dict(
        title=dict(
            text="Borough",
            font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
        ),
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        title=dict(
            text="Total Arrests per ZIP Code (2025)",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    yaxis=dict(
        title=dict(
            text="Median Property Sale Price",
            font=dict(family=THEME["font_family"], size=THEME["axis_title_size"], color=THEME["font_color"])
        ),
        tickprefix="$", tickformat=",.0f",
        tickfont=dict(family=THEME["font_family"], size=13, color=THEME["font_color"]),
        gridcolor=THEME["gridcolor"], linecolor=THEME["linecolor"], showline=True
    ),
    hoverlabel=dict(
        bgcolor="white", bordercolor="rgba(0,0,0,0.15)",
        font=dict(family=THEME["font_family"], size=13, color=THEME["font_color"])
    )
)

fig_scatter_crime.show()

✅ scatter_crime_data: (180, 6)
  zip_code    borough  median_sale_price  total_sales  arrest_count  \
0    10001  manhattan          2487500.0         1793           332
1    10002  manhattan          1531044.5         2434          2441
2    10003  manhattan          2195000.0         2543          2070
3    10004  manhattan          1149095.0         1071            81
4    10005  manhattan          1127500.0         1751             6

  borough_display
0       Manhattan
1       Manhattan
2       Manhattan
3       Manhattan
4       Manhattan


In [127]:
# ============================================================
# CORRELATION ANALYSIS - which factors relate most to price?
# ============================================================

zip_summary = (
    df_master.groupby("zip_code")
    .agg(median_price=("sale_price", "median"))
    .reset_index()
)

# Merge — using actual column names
zip_summary = zip_summary.merge(df_income[["zip_code", "median_income_usd"]], on="zip_code", how="left")
zip_summary = zip_summary.merge(df_sch_score[["zip_code", "school_score"]], on="zip_code", how="left")
zip_summary = zip_summary.merge(arrest_zip_enhanced_clean[["zip_code", "crime_per_10k", "murder", "robbery", "burglary", "gr_larceny"]], on="zip_code", how="left")

# Correlation
factors = ["median_income_usd", "school_score", "crime_per_10k", "murder", "robbery", "burglary", "gr_larceny"]
correlations = zip_summary[["median_price"] + factors].corr()["median_price"].drop("median_price")

print("📊 Correlation with Median Sale Price:")
print("=" * 55)
for factor, corr in correlations.sort_values(key=abs, ascending=False).items():
    bar = "█" * int(abs(corr) * 30)
    direction = "↑" if corr > 0 else "↓"
    print(f"{factor:25s} {direction} {corr:+.3f}  {bar}")

📊 Correlation with Median Sale Price:
gr_larceny                ↑ +0.602  ██████████████████
median_income_usd         ↑ +0.413  ████████████
school_score              ↑ +0.397  ███████████
burglary                  ↑ +0.371  ███████████
crime_per_10k             ↑ +0.264  ███████
murder                    ↓ -0.176  █████
robbery                   ↑ +0.068  ██


In [126]:
print("df_income columns:", df_income.columns.tolist())
print("df_sch_score columns:", df_sch_score.columns.tolist())
print("arrest_zip_enhanced_clean columns:", arrest_zip_enhanced_clean.columns.tolist())
print("df_subway columns:", df_subway.columns.tolist())

df_income columns: ['rank', 'zip_code', 'median_income_usd']
df_sch_score columns: ['zip_code', 'avg_rank', 'school_score', 'best_school_name', 'best_school_type', 'best_school_level']
arrest_zip_enhanced_clean columns: ['zip_code', 'arrest_count', 'lat', 'lon', 'borough', 'crime_per_10k', 'murder', 'rape', 'robbery', 'fel_assault', 'burglary', 'gr_larceny', 'gla', 'total', 'borough_display']
df_subway columns: ['station_id', 'stop_name', 'borough', 'latitude', 'longitude']


In [128]:
# ============================================================
# CONTROLLED CORRELATION - Within Each Borough
# control for borough to check the relationship
# ============================================================

# add borough
zip_summary = zip_summary.merge(
    df_master.groupby("zip_code")["borough"].first().reset_index(),
    on="zip_code", how="left"
)

factors = ["median_income_usd", "school_score", "crime_per_10k", "murder", "burglary", "gr_larceny"]

print("📊 Correlation with Median Sale Price - WITHIN each borough:")
print("=" * 85)
print(f"{'Factor':<22} {'Bronx':>10} {'Brooklyn':>10} {'Manhattan':>10} {'Queens':>10} {'Staten':>10}")
print("-" * 85)

for factor in factors:
    row = f"{factor:<22}"
    for boro in ["bronx", "brooklyn", "manhattan", "queens", "statenisland"]:
        sub = zip_summary[zip_summary["borough"] == boro]
        if len(sub) > 3:
            corr = sub[["median_price", factor]].corr().iloc[0, 1]
            row += f" {corr:>+10.3f}"
        else:
            row += f" {'N/A':>10}"
    print(row)

print("\n💡 Citywide vs. within-borough comparison:")
print("=" * 60)
print(f"{'Factor':<22} {'Citywide':>10} {'Borough avg':>15}")
print("-" * 60)

# Citywide correlation
overall = zip_summary[["median_price"] + factors].corr()["median_price"].drop("median_price")

# average within-borough correlation
for factor in factors:
    boro_corrs = []
    for boro in ["bronx", "brooklyn", "manhattan", "queens", "statenisland"]:
        sub = zip_summary[zip_summary["borough"] == boro]
        if len(sub) > 3:
            corr = sub[["median_price", factor]].corr().iloc[0, 1]
            boro_corrs.append(corr)
    avg_within = sum(boro_corrs) / len(boro_corrs) if boro_corrs else 0
    flip = "⚠️ sign flips" if (overall[factor] > 0 and avg_within < 0) or (overall[factor] < 0 and avg_within > 0) else ""
    print(f"{factor:<22} {overall[factor]:>+10.3f} {avg_within:>+15.3f}  {flip}")

📊 Correlation with Median Sale Price - WITHIN each borough:
Factor                      Bronx   Brooklyn  Manhattan     Queens     Staten
-------------------------------------------------------------------------------------
median_income_usd            +nan     +0.441     +0.007     +0.802     +0.908
school_score               +0.446     +0.425     +0.320     +0.385     -0.468
crime_per_10k                +nan       +nan       +nan       +nan       +nan
murder                       +nan       +nan       +nan       +nan       +nan
burglary                     +nan       +nan       +nan       +nan       +nan
gr_larceny                   +nan       +nan       +nan       +nan       +nan

💡 Citywide vs. within-borough comparison:
Factor                         Citywide       Borough avg
------------------------------------------------------------
median_income_usd          +0.413            +nan
school_score               +0.397          +0.222
crime_per_10k              +0.264            +

In [134]:
# ============================================================
# CLEAN CORRELATION ANALYSIS
# ============================================================

# 1. ZIP-level house price (deduplicated)
zip_price = (
    df_master.groupby("zip_code")["sale_price"].median()
    .reset_index().rename(columns={"sale_price": "median_price"})
)

# 2. borough for each ZIP (most common)
zip_borough = df_master.groupby("zip_code")["borough"].agg(lambda x: x.mode()[0]).reset_index()

# 3. Income per ZIP (deduplicated)
income_clean = df_income.drop_duplicates("zip_code")[["zip_code", "median_income_usd"]]

# 4. School per ZIP (deduplicated)
school_clean = df_sch_score.drop_duplicates("zip_code")[["zip_code", "school_score"]]

# 5. Crime per ZIP (deduplicated)
crime_clean = arrest_zip_enhanced_clean.drop_duplicates("zip_code")[
    ["zip_code", "crime_per_10k", "murder", "robbery", "burglary", "gr_larceny"]
]

# Merge all
zip_clean = (
    zip_price
    .merge(zip_borough, on="zip_code", how="left")
    .merge(income_clean, on="zip_code", how="left")
    .merge(school_clean, on="zip_code", how="left")
    .merge(crime_clean, on="zip_code", how="left")
)

print(f"Total ZIPs: {len(zip_clean)}")
print(f"\nNaN per column:")
print(zip_clean.isna().sum())

print(f"\n📊 Correlation with Median Sale Price (drop NaN):")
print("=" * 55)
factors = ["median_income_usd", "school_score", "crime_per_10k", "murder", "burglary", "gr_larceny"]
for factor in factors:
    sub = zip_clean[["median_price", factor]].dropna()
    if len(sub) > 5:
        corr = sub.corr().iloc[0, 1]
        bar = "█" * int(abs(corr) * 30)
        direction = "↑" if corr > 0 else "↓"
        print(f"{factor:25s} {direction} {corr:+.3f} (n={len(sub):3d})  {bar}")

Total ZIPs: 185

NaN per column:
zip_code               0
median_price           0
borough                0
median_income_usd    131
school_score         116
crime_per_10k          5
murder                 5
robbery                5
burglary               5
gr_larceny             5
dtype: int64

📊 Correlation with Median Sale Price (drop NaN):
median_income_usd         ↑ +0.497 (n= 54)  ██████████████
school_score              ↑ +0.403 (n= 69)  ████████████
crime_per_10k             ↑ +0.281 (n=180)  ████████
murder                    ↓ -0.174 (n=180)  █████
burglary                  ↑ +0.372 (n=180)  ███████████
gr_larceny                ↑ +0.606 (n=180)  ██████████████████


In [135]:
# ============================================================
# VIZ - Crime Type Impact on Property Price
# ============================================================

# Compute correlations for crime types only
crime_types = ["murder", "robbery", "burglary", "gr_larceny"]
crime_labels = {
    "murder": "Murder",
    "robbery": "Robbery",
    "burglary": "Burglary",
    "gr_larceny": "Grand Larceny"
}

corr_data = []
for crime in crime_types:
    sub = zip_clean[["median_price", crime]].dropna()
    corr = sub.corr().iloc[0, 1]
    corr_data.append({
        "crime": crime_labels[crime],
        "correlation": corr,
        "abs_corr": abs(corr),
    })

corr_df = pd.DataFrame(corr_data).sort_values("correlation")

# Color: red for negative, green for positive
corr_df["color"] = corr_df["correlation"].apply(
    lambda x: "#EF553B" if x < 0 else "#636EFA"
)

fig_crime_impact = go.Figure()

fig_crime_impact.add_trace(go.Bar(
    y=corr_df["crime"],
    x=corr_df["correlation"],
    orientation="h",
    marker=dict(color=corr_df["color"]),
    text=corr_df["correlation"].apply(lambda x: f"{x:+.3f}"),
    textposition="outside",
    textfont=dict(size=14, family="Arial"),
))

# Vertical line at 0
fig_crime_impact.add_vline(x=0, line=dict(color="#999", width=1, dash="dash"))

fig_crime_impact.update_layout(
    title=dict(
        text="<b>Which Crimes Actually Lower Property Prices?</b><br><sub>Correlation with Median Sale Price by ZIP Code (n=180)</sub>",
        x=0.5, xanchor="center",
        font=dict(family="Arial", size=18, color="#2c2c2c"),
    ),
    xaxis=dict(
        title="Correlation Coefficient",
        range=[-0.3, 0.7],
        zeroline=False,
        gridcolor="#e8e8e8",
        tickfont=dict(family="Arial", size=12),
    ),
    yaxis=dict(
        title="",
        tickfont=dict(family="Arial", size=14),
    ),
    plot_bgcolor="white",
    height=400,
    margin={"l": 120, "r": 80, "t": 100, "b": 60},
    annotations=[
        dict(
            x=-0.20, y=-0.5, xref="x", yref="y",
            text="<i>Lowers price ↓</i>",
            showarrow=False,
            font=dict(color="#EF553B", size=11, family="Arial"),
        ),
        dict(
            x=0.45, y=-0.5, xref="x", yref="y",
            text="<i>Reflects busy/dense areas (Manhattan effect)</i>",
            showarrow=False,
            font=dict(color="#636EFA", size=11, family="Arial"),
        ),
    ],
)

fig_crime_impact.show()

In [139]:
# property type distribution
print("Top property types:")
print(df_master["building_class_category"].value_counts().head(15))

print("\nSimplified categories after cleaning:")
# take the first 2 digits of the code
df_master["building_code"] = df_master["building_class_category"].astype(str).str[:2].str.strip()
print(df_master["building_code"].value_counts().head(10))

Top property types:
building_class_category
01 ONE FAMILY DWELLINGS                       150793
13 CONDOS - ELEVATOR APARTMENTS               135826
02 TWO FAMILY DWELLINGS                       114519
03 THREE FAMILY DWELLINGS                      28240
07 RENTALS - WALKUP APARTMENTS                 16356
04 TAX CLASS 1 CONDOS                          15671
15 CONDOS - 2-10 UNIT RESIDENTIAL              15414
12 CONDOS - WALKUP APARTMENTS                   9982
14 RENTALS - 4-10 UNIT                          4567
10 COOPS - ELEVATOR APARTMENTS                  3600
17 CONDO COOPS                                  1121
09 COOPS - WALKUP APARTMENTS                    1070
16 CONDOS - 2-10 UNIT WITH COMMERCIAL UNIT       600
08 RENTALS - ELEVATOR APARTMENTS                 207
11A CONDO-RENTALS                                 17
Name: count, dtype: int64

Simplified categories after cleaning:
building_code
01    150793
13    135826
02    114519
03     28240
07     16356
04     15671
15  

In [140]:
# ============================================================
# Test: Group property types into 5 simple categories
# ============================================================

def classify_property(building_class):
    """Group 50+ NYC property types into 5 main categories"""
    if pd.isna(building_class):
        return "Other"
    code = str(building_class).strip()[:2]
    mapping = {
        "01": "1-Family",
        "02": "2-Family",
        "03": "3-Family",
        "13": "Condo",
        "15": "Condo",
        "12": "Condo",
        "04": "Condo",
        "16": "Condo",
        "17": "Condo",
        "07": "Rental",
        "08": "Rental",
        "14": "Rental",
        "11": "Rental",
        "09": "Co-op",
        "10": "Co-op",
    }
    return mapping.get(code, "Other")

df_master["property_type"] = df_master["building_class_category"].apply(classify_property)

# Test the result
print("📊 Distribution of simplified categories:")
print(df_master["property_type"].value_counts())

print("\n💰 Median price by property type:")
print(df_master.groupby("property_type")["sale_price"].median().sort_values(ascending=False))

print("\n💰 Median price by property type AND borough:")
print(df_master.groupby(["property_type", "borough"])["sale_price"].median().unstack().round(0))

📊 Distribution of simplified categories:
property_type
Condo       178614
1-Family    150793
2-Family    114519
3-Family     28240
Rental       21147
Co-op         4670
Other           17
Name: count, dtype: int64

💰 Median price by property type:
property_type
Rental      1325000.0
Condo        950000.0
3-Family     925000.0
2-Family     750000.0
1-Family     600000.0
Co-op        342250.0
Other         65000.0
Name: sale_price, dtype: float64

💰 Median price by property type AND borough:
borough           bronx   brooklyn  manhattan     queens  statenisland
property_type
1-Family       480000.0   740000.0  6887500.0   648000.0      525000.0
2-Family       565000.0   900000.0  3862500.0   750000.0      670000.0
3-Family       650000.0  1050000.0  2525650.0   975000.0      570000.0
Co-op          195000.0   570336.0   650000.0   318000.0      230000.0
Condo          215000.0   850238.0  1595000.0   590585.0      340000.0
Other               NaN  1150000.0    64441.0        NaN         

In [141]:
# ============================================================
# VIZ - Sale Price by Property Type & Borough
# ============================================================

dff = df_master[df_master["property_type"] != "Other"].copy()
dff["borough_display"] = dff["borough"].map(BOROUGH_NAME_MAP)
price_cap = dff["sale_price"].quantile(0.99)
dff = dff[dff["sale_price"] <= price_cap]

# Order property types by median price (high to low)
type_order = (
    dff.groupby("property_type")["sale_price"].median()
    .sort_values(ascending=False).index.tolist()
)

fig_type = px.box(
    dff,
    x="property_type",
    y="sale_price",
    color="borough_display",
    color_discrete_map=THEME["borough_colors"],
    category_orders={"property_type": type_order},
    title="Sale Price by Property Type & Borough (2013-2025)",
    labels={"property_type": "Property Type", "sale_price": "Sale Price", "borough_display": "Borough"},
    points=False,
)

fig_type.update_layout(
    title=dict(
        x=0.5, xanchor="center",
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"])
    ),
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    xaxis=dict(title="Property Type"),
    yaxis=dict(
        title="Sale Price",
        tickprefix="$",
        tickformat=",.0f",
        gridcolor=THEME["gridcolor"],
    ),
    legend=dict(title="Borough", orientation="v"),
    plot_bgcolor=THEME["plot_bgcolor"],
    paper_bgcolor=THEME["paper_bgcolor"],
    height=THEME["height_medium"],
    margin=THEME["margin_chart"],
    boxmode="group",  # grouped side by side
)

fig_type.show()

In [142]:
# ============================================================
# VIZ - Monthly Seasonality: Volume + Price
# ============================================================

# prepare monthly data
df_master["sale_month"] = pd.to_datetime(df_master["sale_date"]).dt.month

monthly = (
    df_master.groupby("sale_month")
    .agg(
        median_price=("sale_price", "median"),
        transaction_count=("sale_price", "count"),
    )
    .reset_index()
)

month_labels = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}
monthly["month_name"] = monthly["sale_month"].map(month_labels)

# Dual-axis chart: Bar = transactions, Line = median price
fig_seasonal = go.Figure()

# Bar: Transaction count (left axis)
fig_seasonal.add_trace(go.Bar(
    x=monthly["month_name"],
    y=monthly["transaction_count"],
    name="Transactions",
    marker=dict(color="#A0AEC0", opacity=0.6),
    yaxis="y",
    hovertemplate="<b>%{x}</b><br>Transactions: %{y:,}<extra></extra>",
))

# Line: Median price (right axis)
fig_seasonal.add_trace(go.Scatter(
    x=monthly["month_name"],
    y=monthly["median_price"],
    name="Median Price",
    mode="lines+markers",
    line=dict(color="#00CC96", width=3),
    marker=dict(size=10, color="#00CC96"),
    yaxis="y2",
    hovertemplate="<b>%{x}</b><br>Median: $%{y:,.0f}<extra></extra>",
))

fig_seasonal.update_layout(
    title=dict(
        text="When is the Best Time to Buy in NYC?<br><sub>Monthly seasonality: transaction volume vs. median price (2013–2025)</sub>",
        x=0.5, xanchor="center",
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"]),
    ),
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    xaxis=dict(
        title="Month",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        title="Number of Transactions",
        side="left",
        gridcolor=THEME["gridcolor"],
        tickformat=",",
    ),
    yaxis2=dict(
        title="Median Sale Price",
        side="right",
        overlaying="y",
        tickprefix="$",
        tickformat=",.0f",
        showgrid=False,
    ),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#e8e8e8",
        borderwidth=1,
    ),
    plot_bgcolor=THEME["plot_bgcolor"],
    paper_bgcolor=THEME["paper_bgcolor"],
    height=THEME["height_medium"],
    margin=dict(t=100, b=60, l=80, r=80),
    hovermode="x unified",
)

fig_seasonal.show()

# Print insight
print("\n📊 Seasonal Insights:")
peak_volume_month = monthly.loc[monthly["transaction_count"].idxmax(), "month_name"]
peak_price_month = monthly.loc[monthly["median_price"].idxmax(), "month_name"]
low_volume_month = monthly.loc[monthly["transaction_count"].idxmin(), "month_name"]
low_price_month = monthly.loc[monthly["median_price"].idxmin(), "month_name"]
print(f"📈 Peak transaction volume:  {peak_volume_month}")
print(f"💰 Highest median price:     {peak_price_month}")
print(f"📉 Lowest transaction volume: {low_volume_month}")
print(f"🏷️  Lowest median price:      {low_price_month}")


📊 Seasonal Insights:
📈 Peak transaction volume:  Jun
💰 Highest median price:     Jun
📉 Lowest transaction volume: Feb
🏷️  Lowest median price:      Feb


In [146]:
# ============================================================
# VIZ - Monthly Seasonality with Borough Breakdown + Annotations
# ============================================================

df_master["sale_month"] = pd.to_datetime(df_master["sale_date"]).dt.month
month_labels = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
                7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"}

# Overall monthly stats
monthly_total = (
    df_master.groupby("sale_month")
    .agg(median_price=("sale_price", "median"),
         transaction_count=("sale_price", "count"))
    .reset_index()
)
monthly_total["month_name"] = monthly_total["sale_month"].map(month_labels)

# Per-borough monthly median
monthly_boro = (
    df_master.groupby(["sale_month", "borough"])["sale_price"]
    .median().reset_index()
    .rename(columns={"sale_price": "median_price"})
)
monthly_boro["month_name"] = monthly_boro["sale_month"].map(month_labels)
monthly_boro["borough_display"] = monthly_boro["borough"].map(BOROUGH_NAME_MAP)
month_order = list(month_labels.values())

# Build figure
fig_seasonal = go.Figure()

# Bar: Total transaction count (left axis, gray, low opacity)
fig_seasonal.add_trace(go.Bar(
    x=monthly_total["month_name"],
    y=monthly_total["transaction_count"],
    name="Transactions (NYC total)",
    marker=dict(color="#CBD5E0", opacity=0.5),
    yaxis="y",
    hovertemplate="<b>%{x}</b><br>Transactions: %{y:,}<extra></extra>",
))

# Line: Per-borough median price (right axis)
for boro_display, color in THEME["borough_colors"].items():
    sub = monthly_boro[monthly_boro["borough_display"] == boro_display]
    fig_seasonal.add_trace(go.Scatter(
        x=sub["month_name"],
        y=sub["median_price"],
        name=boro_display,
        mode="lines+markers",
        line=dict(color=color, width=2.5),
        marker=dict(size=7),
        yaxis="y2",
        hovertemplate=f"<b>{boro_display}</b><br>%{{x}}<br>Median: $%{{y:,.0f}}<extra></extra>",
    ))

# Find peak/low for annotations
peak_month = monthly_total.loc[monthly_total["transaction_count"].idxmax(), "month_name"]
low_month = monthly_total.loc[monthly_total["transaction_count"].idxmin(), "month_name"]

fig_seasonal.update_layout(
    title=dict(
        text="When is the Best Time to Buy in NYC?<br>"
             "<sub>Monthly transaction volume + median price by borough (2013–2025)</sub>",
        x=0.5, xanchor="center",
        font=dict(family=THEME["font_family"], size=THEME["title_size"], color=THEME["font_color"]),
    ),
    font=dict(family=THEME["font_family"], size=THEME["font_size"], color=THEME["font_color"]),
    xaxis=dict(
        title="Month",
        categoryorder="array",
        categoryarray=month_order,
    ),
    yaxis=dict(
        title="Number of Transactions",
        side="left",
        gridcolor=THEME["gridcolor"],
        tickformat=",",
    ),
    yaxis2=dict(
        title="Median Sale Price (by borough)",
        side="right",
        overlaying="y",
        tickprefix="$",
        tickformat=",.0f",
        showgrid=False,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.25,
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#e8e8e8",
        borderwidth=1,
    ),
    plot_bgcolor=THEME["plot_bgcolor"],
    paper_bgcolor=THEME["paper_bgcolor"],
    height=550,
    margin=dict(t=110, b=130, l=80, r=80),
    hovermode="x unified",
    annotations=[
        dict(
            x=low_month, y=monthly_total[monthly_total["month_name"]==low_month]["transaction_count"].iloc[0],
            xref="x", yref="y",
            text="💰 <b>Best for Buyers</b><br>Lowest volume + price",
            showarrow=True, arrowhead=2, arrowcolor="#2D9CDB",
            ax=-30, ay=-60,
            bgcolor="rgba(45,156,219,0.1)",
            bordercolor="#2D9CDB", borderwidth=1.5,
            font=dict(color="#2D9CDB", size=11, family="Arial"),
        ),
        dict(
            x=peak_month, y=monthly_total[monthly_total["month_name"]==peak_month]["transaction_count"].iloc[0],
            xref="x", yref="y",
            text="🔥 <b>Peak Market</b><br>Most competitive",
            showarrow=True, arrowhead=2, arrowcolor="#EF553B",
            ax=30, ay=-60,
            bgcolor="rgba(239,85,59,0.1)",
            bordercolor="#EF553B", borderwidth=1.5,
            font=dict(color="#EF553B", size=11, family="Arial"),
        ),
    ],
)

fig_seasonal.show()

In [157]:
# ============================================================
# KPI Cards for Monthly Seasonality (Filter-Ready)
# ============================================================

monthly_total_overall = (
    df_master.groupby("sale_month")
    .agg(median_price=("sale_price", "median"),
         transaction_count=("sale_price", "count"))
    .reset_index()
)
month_labels = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
                7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"}
monthly_total_overall["month_name"] = monthly_total_overall["sale_month"].map(month_labels)

# Compute KPI values
low_price_month  = monthly_total_overall.loc[monthly_total_overall["median_price"].idxmin()]
high_price_month = monthly_total_overall.loc[monthly_total_overall["median_price"].idxmax()]
peak_month_data  = monthly_total_overall.loc[monthly_total_overall["transaction_count"].idxmax()]
low_month_data   = monthly_total_overall.loc[monthly_total_overall["transaction_count"].idxmin()]

spread_pct = (monthly_total_overall["median_price"].max() - monthly_total_overall["median_price"].min()) / monthly_total_overall["median_price"].min() * 100
volume_drop_pct = (peak_month_data["transaction_count"] - low_month_data["transaction_count"]) / peak_month_data["transaction_count"] * 100

# Build figure
fig_kpi = go.Figure()

kpis = [
    {"label": "LOWEST MEDIAN PRICE", "value": low_price_month["month_name"],   "sub": f"${low_price_month['median_price']/1000:.0f}K median",                  "color": "#2D9CDB"},
    {"label": "HIGHEST VOLUME",      "value": peak_month_data["month_name"],   "sub": f"{peak_month_data['transaction_count']:,} transactions",                "color": "#EF553B"},
    {"label": "PRICE RANGE",         "value": f"+{spread_pct:.1f}%",            "sub": f"{low_price_month['month_name']} → {high_price_month['month_name']}",   "color": "#00CC96"},
    {"label": "VOLUME RANGE",        "value": f"-{volume_drop_pct:.0f}%",       "sub": f"{low_month_data['month_name']} vs {peak_month_data['month_name']}",    "color": "#AB63FA"},
]

fig_kpi.update_layout(
    height=200,
    margin=dict(t=15, b=15, l=20, r=20),
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(visible=False, range=[0, 4]),
    yaxis=dict(visible=False, range=[0, 1]),
    showlegend=False,
    annotations=[
        ann
        for i, kpi in enumerate(kpis)
        for ann in [
            dict(
                x=i + 0.5, y=0.85, xref="x", yref="paper",
                text=kpi["label"],
                showarrow=False,
                font=dict(size=18, color="#444", family="Arial"),
                xanchor="center",
            ),
            dict(
                x=i + 0.5, y=0.5, xref="x", yref="paper",
                text=f"<b>{kpi['value']}</b>",
                showarrow=False,
                font=dict(size=48, color=kpi["color"], family="Arial"),
                xanchor="center",
            ),
            dict(
                x=i + 0.5, y=0.13, xref="x", yref="paper",
                text=kpi["sub"],
                showarrow=False,
                font=dict(size=13, color="#888", family="Arial"),
                xanchor="center",
            ),
        ]
    ],
    shapes=[
        dict(
            type="rect",
            x0=i + 0.04, x1=i + 0.96,
            y0=0.02, y1=1.0,
            xref="x", yref="paper",
            fillcolor="#fafbfc",
            line=dict(color="#e1e4e8", width=1),
            layer="below",
        ) for i in range(4)
    ] + [
        dict(
            type="rect",
            x0=i + 0.04, x1=i + 0.96,
            y0=0.96, y1=1.0,
            xref="x", yref="paper",
            fillcolor=kpis[i]["color"],
            line=dict(width=0),
        ) for i in range(4)
    ],
)

fig_kpi.show()

In [158]:
import pandas as pd

# read the full backup
df_full = pd.read_csv("Datasets/df_master_full_backup.csv.gz")

# check the columns
print("Columns:", df_full.columns.tolist())
print("Has sale_date?", "sale_date" in df_full.columns)

Columns: ['borough', 'neighborhood', 'building_class_category', 'tax_class_at_present', 'block', 'lot', 'building_class_at_present', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'land_square_feet', 'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale', 'building_class_at_time_of_sale', 'sale_price', 'sale_date', 'year', 'median_income_usd', 'avg_school_rank', 'subway_station_count', 'arrest_count_2025', 'cs_murder_2025', 'cs_rape_2025', 'cs_robbery_2025', 'cs_fel_assault_2025', 'cs_burglary_2025', 'cs_gr_larceny_2025', 'cs_gla_2025', 'cs_total_2025']
Has sale_date? True


/var/folders/sy/y1hmbmcd2hbg9k_571sx10880000gn/T/ipykernel_1424/4277769487.py:4: DtypeWarning:

Columns (0: tax_class_at_present, 1: building_class_at_present) have mixed types. Specify dtype option on import or set low_memory=False.



In [159]:
import pandas as pd

# read the full backup
df_full = pd.read_csv("Datasets/df_master_full_backup.csv.gz", low_memory=False)

# add sale_month column
df_full["sale_month"] = pd.to_datetime(
    df_full["sale_date"], errors="coerce"
).dt.month.astype("Int8")  # Int8 saves space (1 to 12 fits in 1 byte)

# keep only the columns the dashboard uses
keep_cols = [
    "borough",
    "zip_code",
    "year",
    "sale_month",        # new
    "sale_price",
    "building_class_category",  # kept for a later property type filter
]

df_slim = df_full[keep_cols].copy()

# save (overwrites df_master.csv.gz)
df_slim.to_csv("Datasets/df_master.csv.gz",
               index=False, compression="gzip")

# verify
print(f"✅ Saved with {len(df_slim):,} rows")
print(f"Columns: {df_slim.columns.tolist()}")
print(f"Has sale_month? {'sale_month' in df_slim.columns}")
print(f"\nSample:")
print(df_slim.head(3))

# check file size
import os
size_mb = os.path.getsize("Datasets/df_master.csv.gz") / 1024 / 1024
print(f"\nFile size: {size_mb:.1f} MB")

✅ Saved with 498,000 rows
Columns: ['borough', 'zip_code', 'year', 'sale_month', 'sale_price', 'building_class_category']
Has sale_month? True

Sample:
  borough  zip_code  year  sale_month  sale_price  building_class_category
0   bronx   10457.0  2013           7    355000.0  01 ONE FAMILY DWELLINGS
1   bronx   10458.0  2013          11    120000.0  01 ONE FAMILY DWELLINGS
2   bronx   10458.0  2013           5    474819.0  01 ONE FAMILY DWELLINGS

File size: 2.1 MB


In [160]:
import shutil
shutil.copy(
    "Datasets/df_master.csv.gz",
    "Datasets/df_master_backup_before_property.csv.gz"
)
print("✅ Backup saved")

✅ Backup saved


In [161]:
import pandas as pd

df_full = pd.read_csv("Datasets/df_master_full_backup.csv.gz", low_memory=False)

# Add sale_month
df_full["sale_month"] = pd.to_datetime(
    df_full["sale_date"], errors="coerce"
).dt.month.astype("Int8")

# Add property_type
def classify_property(building_class):
    if pd.isna(building_class):
        return "Other"
    code = str(building_class).strip()[:2]
    mapping = {
        "01": "1-Family", "02": "2-Family", "03": "3-Family",
        "13": "Condo", "15": "Condo", "12": "Condo",
        "04": "Condo", "16": "Condo", "17": "Condo",
        "07": "Rental", "08": "Rental", "14": "Rental", "11": "Rental",
        "09": "Co-op", "10": "Co-op",
    }
    return mapping.get(code, "Other")

df_full["property_type"] = df_full["building_class_category"].apply(classify_property)

# Keep only needed columns
keep_cols = [
    "borough", "zip_code", "year", "sale_month",
    "sale_price", "property_type"
]
df_slim = df_full[keep_cols].copy()

# Save
df_slim.to_csv("Datasets/df_master.csv.gz", index=False, compression="gzip")

# Verify
print(f"✅ Saved {len(df_slim):,} rows")
print(f"Columns: {df_slim.columns.tolist()}")
print(f"\nProperty type distribution:")
print(df_slim["property_type"].value_counts())

import os
size_mb = os.path.getsize("Datasets/df_master.csv.gz") / 1024 / 1024
print(f"\nFile size: {size_mb:.1f} MB")

✅ Saved 498,000 rows
Columns: ['borough', 'zip_code', 'year', 'sale_month', 'sale_price', 'property_type']

Property type distribution:
property_type
Condo       178614
1-Family    150793
2-Family    114519
3-Family     28240
Rental       21147
Co-op         4670
Other           17
Name: count, dtype: int64

File size: 2.0 MB


In [ ]:
## Dashboard

## Dashboard

## save file

In [115]:
import os
os.makedirs("Datasets", exist_ok=True)

df_master.to_csv("Datasets/df_master.csv", index=False)
df_schools_all.to_csv("Datasets/df_schools_all.csv", index=False)
df_income.to_csv("Datasets/df_income.csv", index=False)
df_subway.to_csv("Datasets/df_subway.csv", index=False)
arrest_per_zip.to_csv("Datasets/arrest_per_zip.csv", index=False)
df_zip_centroids.to_csv("Datasets/df_zip_centroids.csv", index=False)
arrest_zip_enhanced_clean.to_csv("Datasets/arrest_zip_enhanced.csv", index=False)

print("✅ All datasets saved!")

✅ All datasets saved!
